In [ ]:
# ============================================================
# Sweden-USA common-feature CNN for PMA prediction
#
# Stage A: audit the exact Sweden-USA common feature set
# Stage B: create/reuse a 120-min / 40-min common-feature PKL
# Stage C: rerun publication-grade patient-level nested CV
#          5 outer folds x 3 inner folds x 100 Optuna trials
#
# Final outer model: full outer-train refit with a 3-seed ensemble
# Dynamic normalization: patient-balanced, per-feature z-score using training data only
# Training: patient-balanced window sampling
# Scheduler: validation-independent cosine schedule, identical in inner/final training
# Resume: Optuna top-up + outer-fold/seed/epoch recovery
# Outer folds: reused exactly from the completed full-feature CNN experiment
# Leakage control: pna_days is target-construction-only and never enters the model
# ============================================================

import os
import glob
import json
import pickle
import random
import gc
import re
import warnings
import hashlib
import math
from pathlib import Path
from collections import OrderedDict, Counter

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import optuna
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score

warnings.filterwarnings("default")

# ============================================================
# 0. Config
# ============================================================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
PROJECT_ROOT = "/mnt/home/qinqiu/thesis"

# Existing full-feature 120-min / 40-min PKL.
SOURCE_DATASET_ROOT = os.path.join(
    PROJECT_ROOT,
    "data",
    "superwindow_stride_comparison_90_120_3h_no_repeated_weight_no_los_nec_ga_le_35_no_intubation_imputed_2ch_CTHW",
)
SOURCE_DATASET_LABEL = "120min_stride40min_overlap67"

# Optional: set this to the exact full-feature PKL path to bypass auto-discovery.
SOURCE_DATA_PKL_OVERRIDE = None

# The common-feature dataset is materialized once and then reused.
COMMON_DATASET_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "sweden_usa_common_feature_120min_stride40min",
)
os.makedirs(COMMON_DATASET_DIR, exist_ok=True)
COMMON_DATA_PKL = os.path.join(
    COMMON_DATASET_DIR,
    "sweden_120min_stride40min_usa_common17_imputed_2ch_CTHW.pkl",
)
REBUILD_COMMON_DATASET = False
# Set True for a first audit-only run; the script exits after saving the common PKL and checklists.
STOP_AFTER_COMMON_DATASET_AND_AUDIT = False

# Completed full-feature CNN experiment. Its outer-test assignments are reused
# exactly, so full-feature and common-feature CNN results are paired fairly.
MAIN_FULL_CNN_EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_nested",
    "nested_cv_120min_stride40min_v7_dataset_reuse_patientbalanced_resume100",
)
MAIN_OUTER_ASSIGNMENT_CSV = os.path.join(
    MAIN_FULL_CNN_EXP_ROOT,
    "outer_fold_assignment_long.csv",
)
MAIN_FULL_CNN_PATIENT_OOF_CSV = os.path.join(
    MAIN_FULL_CNN_EXP_ROOT,
    "FINAL_nested_oof_patient_predictions.csv",
)

# New output path: never mix this run with the full-feature experiment.
EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_nested",
    "nested_cv_120min_stride40min_usa_common17_v1_patientbalanced_resume100",
)
os.makedirs(EXP_ROOT, exist_ok=True)

# Optional numerical cross-cohort audit.
# Set USA_TABULAR_PATH to a CSV, Parquet, Feather, or PKL containing a pandas
# DataFrame with the USA columns below. This is NOT required to train on Sweden.
USA_TABULAR_PATH = None

# ------------------------------------------------------------------
# Cross-validation / optimization
# ------------------------------------------------------------------
OUTER_SPLITS = 5
INNER_SPLITS = 3
N_TRIALS_PER_OUTER = 100      # formal run; use 2-5 only for a smoke test
MAX_EPOCHS = 300
N_WORKERS = 2
PIN_MEMORY = DEVICE.type == "cuda"

# Each final outer model is trained on the complete outer-train set.
FINAL_REFIT_SEEDS = [42, 123, 2026]

SAVE_BEST_CHECKPOINTS = True
SAVE_TRIAL_CHECKPOINTS = False
RUN_POST_ANALYSIS = True

CODE_VERSION = "v1_usa_common17_exact_main_outer_folds_resume100"
PATIENT_BALANCED_TRAINING = True
PATIENT_BALANCED_NORMALIZATION = True
PRED_BATCH_SIZE = 512
CHECKPOINT_EVERY_EPOCH = True

# ------------------------------------------------------------------
# Exact Sweden-USA common feature contract
# ------------------------------------------------------------------
COMMON_DYNAMIC_FEATURES = [
    "feats__spo2_mean",
    "feats__spo2_std",
    "feats__spo2_max",
    "feats__spo2_min",
    "feats__spo2_skew",
    "feats__spo2_kurtosis",
    "feats__btb_mean",
    "feats__btb_std",
    "feats__btb_max",
    "feats__btb_min",
    "feats__btb_skew",
    "feats__btb_kurtosis",
    "feats__btb_sampAs",
    "feats__btb_sampEn",
]

STATIC_FEATURES = [
    "feats__ga_w",
    "feats__sex",
    "feats__bw",
]

# USA columns that are intentionally NOT model inputs.
TARGET_ONLY_FEATURES = ["feats__pna_days"]
EXCLUDED_USA_FEATURES = ["feats__apgar_1", "feats__apgar_5"]

USA_FEATURES_REPORTED = [
    *COMMON_DYNAMIC_FEATURES,
    "feats__bw",
    "feats__sex",
    "feats__pna_days",
    "feats__ga_w",
    "feats__apgar_1",
    "feats__apgar_5",
]

# Exact ordered model input contract stored in the common PKL.
SELECTED_FEATURES_ORDERED = COMMON_DYNAMIC_FEATURES + STATIC_FEATURES

# Any occurrence of these features in the final selected model input is fatal.
FORBIDDEN_FEATURES = [
    "feats__weight",      # repeated/current weight
    "feats__pna_days",    # PMA target construction only
    "feats__apgar_1",     # excluded from common model
    "feats__apgar_5",     # excluded from common model
]
STRICT_NO_FORBIDDEN_FEATURES = True
NORMALIZATION_MODE = (
    "patient_balanced_per_dynamic_feature_zscore_training_fold_only"
)

# The script can confirm names, order, ranges and probable scale. It cannot prove
# that two hospitals used identical BTB preprocessing, artifact removal or
# entropy parameters. Review the generated manual checklist before publication.
FEATURE_AUDIT_MAX_VALUES_PER_FEATURE = 500_000

# Strictly select ceil(N * 0.10) patients with the largest absolute OOF error.
OUTLIER_TOP_FRACTION = 0.10

ARCH_CONFIGS = {
    # baseline: static covariates only, no CNN signal branch
    "B0_static_only_hidden_head": {
        "channels": [],
        "use_mid_pool": False,
        "head_type": "hidden",
    },
    # low to moderate CNNs
    "A1_8_16_pool_simple_head": {
        "channels": [8, 16],
        "use_mid_pool": True,
        "head_type": "simple",
    },
    "A2_16_32_pool_simple_head": {
        "channels": [16, 32],
        "use_mid_pool": True,
        "head_type": "simple",
    },
    "A4_32_64_pool_simple_head": {
        "channels": [32, 64],
        "use_mid_pool": True,
        "head_type": "simple",
    },
    "A5_16_32_pool_hidden_head": {
        "channels": [16, 32],
        "use_mid_pool": True,
        "head_type": "hidden",
    },
    "A6_16_32_64_pool_simple_head": {
        "channels": [16, 32, 64],
        "use_mid_pool": True,
        "head_type": "simple",
    },
}

# Static-only remains defined for a separate baseline analysis, but it is
# intentionally excluded from the primary signal-model architecture search.
OPTUNA_ARCHITECTURES = [
    name
    for name in ARCH_CONFIGS
    if not name.startswith("B0_")
]

# ============================================================
# 1. Reproducibility utilities
# ============================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (Path,)):
        return str(value)
    return value


def stable_hash(payload) -> str:
    encoded = json.dumps(
        json_safe(payload),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def atomic_json_dump(payload, path):
    path = os.path.abspath(path)
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, indent=2, sort_keys=True)
    os.replace(tmp_path, path)


def atomic_torch_save(payload, path):
    path = os.path.abspath(path)
    tmp_path = path + ".tmp"
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)


def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


seed_everything(SEED)
print("Normalization mode:", NORMALIZATION_MODE)

# ============================================================
# 2. Audit, build, save and load the common-feature dataset
# ============================================================
def find_dataset_file(dataset_root: str, label: str) -> str:
    patterns = [
        os.path.join(dataset_root, f"*{label}*imputed*2ch*CTHW*.pkl"),
        os.path.join(dataset_root, f"*{label}*2ch*CTHW*.pkl"),
        os.path.join(dataset_root, f"*{label}*imputed*.pkl"),
        os.path.join(dataset_root, f"*{label}*.pkl"),
    ]
    matches = []
    for pattern in patterns:
        matches.extend(glob.glob(pattern))
    matches = list(OrderedDict.fromkeys(matches))
    if not matches:
        raise FileNotFoundError(
            f"Could not find source dataset for label={label} under {dataset_root}."
        )

    def score_path(path):
        name = os.path.basename(path).lower()
        score = 0
        if "imputed" in name:
            score += 10
        if "2ch" in name:
            score += 10
        if "cthw" in name:
            score += 10
        if "loose80" in name:
            score += 3
        if "no_repeated_weight" in path.lower():
            score += 20
        return score

    scored = [(score_path(path), os.path.abspath(path)) for path in matches]
    best_score = max(score for score, _ in scored)
    best = sorted(path for score, path in scored if score == best_score)
    if len(best) != 1:
        raise RuntimeError(
            "Source dataset selection is ambiguous. Set SOURCE_DATA_PKL_OVERRIDE. "
            f"Top matches: {best}"
        )
    return best[0]


def atomic_pickle_dump(payload, path):
    path = os.path.abspath(path)
    tmp_path = path + ".tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp_path, path)


def file_fingerprint(path):
    stat = os.stat(path)
    return {
        "absolute_path": os.path.abspath(path),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def normalize_pid_text(value):
    text_value = str(value).strip()
    if re.fullmatch(r"[-+]?\d+\.0+", text_value):
        text_value = text_value.split(".", 1)[0]
    return text_value


def load_optional_usa_dataframe(path):
    if path is None:
        return None
    path = os.path.abspath(path)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"USA_TABULAR_PATH does not exist: {path}")
    suffix = Path(path).suffix.lower()
    if suffix in {".csv", ".csv.gz"}:
        obj = pd.read_csv(path)
    elif suffix in {".parquet", ".pq"}:
        obj = pd.read_parquet(path)
    elif suffix in {".feather"}:
        obj = pd.read_feather(path)
    elif suffix in {".pkl", ".pickle"}:
        obj = pd.read_pickle(path)
    else:
        raise ValueError(
            "USA_TABULAR_PATH must be CSV, Parquet, Feather, or a pickled DataFrame."
        )
    if not isinstance(obj, pd.DataFrame):
        raise TypeError("USA_TABULAR_PATH did not load as a pandas DataFrame.")
    return obj


def sample_feature_values_from_xdict(
    xdict,
    feature_index,
    max_values=FEATURE_AUDIT_MAX_VALUES_PER_FEATURE,
):
    """Deterministically sample finite value-channel entries without full concatenation."""
    chunks = []
    remaining = int(max_values)
    for pid in sorted(xdict.keys(), key=lambda x: str(x)):
        if remaining <= 0:
            break
        Xp = xdict[pid]
        if Xp is None:
            continue
        arr = np.asarray(Xp)
        if arr.ndim != 4 or arr.shape[1] < 1 or arr.shape[-1] <= feature_index:
            continue
        values = arr[:, 0, :, feature_index].reshape(-1)
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        if values.size > remaining:
            step = max(1, int(np.floor(values.size / remaining)))
            values = values[::step][:remaining]
        chunks.append(values.astype(np.float64, copy=False))
        remaining -= int(values.size)
    if not chunks:
        return np.empty((0,), dtype=np.float64)
    return np.concatenate(chunks)


def expected_role(feature):
    if feature in COMMON_DYNAMIC_FEATURES:
        return "dynamic_model_input"
    if feature in STATIC_FEATURES:
        return "static_model_input"
    if feature in TARGET_ONLY_FEATURES:
        return "target_construction_only"
    if feature in EXCLUDED_USA_FEATURES:
        return "excluded_covariate"
    return "not_declared"


def expected_unit(feature):
    if feature.startswith("feats__spo2_"):
        if feature.endswith(("skew", "kurtosis")):
            return "unitless"
        return "percent (0-100 expected)"
    if feature.startswith("feats__btb_"):
        if feature.endswith(("skew", "kurtosis", "sampAs", "sampEn")):
            return "unitless / algorithm-dependent"
        return "seconds or milliseconds; must match across cohorts"
    return {
        "feats__bw": "grams expected",
        "feats__sex": "categorical coding; must match",
        "feats__pna_days": "days; label construction only",
        "feats__ga_w": "weeks",
        "feats__apgar_1": "score 0-10",
        "feats__apgar_5": "score 0-10",
    }.get(feature, "unknown")


def infer_scale_and_check(feature, values):
    if values.size == 0:
        return "not_available", "no Sweden values found"
    q01, median, q99 = np.quantile(values, [0.01, 0.50, 0.99])
    if feature.startswith("feats__spo2_") and not feature.endswith(("skew", "kurtosis")):
        plausible = q01 >= -1e-6 and q99 <= 100.5
        return "percent_0_100" if plausible else "unexpected_spo2_scale", (
            "plausible" if plausible else "inspect range; expected approximately 0-100"
        )
    if feature == "feats__btb_mean":
        if 0.15 <= median <= 3.0:
            return "likely_seconds", "compare with USA; do not convert unless metadata confirms"
        if 150 <= median <= 3000:
            return "likely_milliseconds", "compare with USA; do not convert unless metadata confirms"
        return "unclear_btb_unit", "manual metadata check required"
    if feature in {"feats__btb_min", "feats__btb_max", "feats__btb_std"}:
        if median < 10:
            return "likely_seconds", "check against btb_mean and USA"
        if median > 50:
            return "likely_milliseconds", "check against btb_mean and USA"
        return "unclear_btb_unit", "manual metadata check required"
    if feature == "feats__bw":
        plausible = q01 > 200 and q99 < 7000
        return "likely_grams" if plausible else "unexpected_birthweight_scale", (
            "plausible" if plausible else "inspect; kilograms vs grams mismatch possible"
        )
    if feature == "feats__ga_w":
        plausible = q01 >= 20 and q99 <= 45
        return "weeks" if plausible else "unexpected_GA_scale", (
            "plausible" if plausible else "inspect gestational-age units"
        )
    if feature == "feats__sex":
        unique = sorted(pd.unique(values).tolist())[:20]
        return "categorical_numeric", f"observed Sweden codes={unique}"
    return "unitless_or_declared", "manual definition check still required"


def build_feature_audit(source_xdict, source_feature_cols, usa_df=None):
    source_index = {name: i for i, name in enumerate(source_feature_cols)}
    rows = []
    for feature in USA_FEATURES_REPORTED:
        values = (
            sample_feature_values_from_xdict(source_xdict, source_index[feature])
            if feature in source_index
            else np.empty((0,), dtype=np.float64)
        )
        inferred_scale, automated_note = infer_scale_and_check(feature, values)
        row = {
            "feature": feature,
            "role_in_common_pipeline": expected_role(feature),
            "expected_unit_or_coding": expected_unit(feature),
            "present_in_sweden_120min_pkl": feature in source_index,
            "selected_as_model_input": feature in SELECTED_FEATURES_ORDERED,
            "sweden_n_sampled_values": int(values.size),
            "sweden_min": float(np.min(values)) if values.size else np.nan,
            "sweden_q01": float(np.quantile(values, 0.01)) if values.size else np.nan,
            "sweden_q05": float(np.quantile(values, 0.05)) if values.size else np.nan,
            "sweden_median": float(np.median(values)) if values.size else np.nan,
            "sweden_q95": float(np.quantile(values, 0.95)) if values.size else np.nan,
            "sweden_q99": float(np.quantile(values, 0.99)) if values.size else np.nan,
            "sweden_max": float(np.max(values)) if values.size else np.nan,
            "sweden_inferred_scale": inferred_scale,
            "automated_audit_note": automated_note,
            "manual_definition_match_confirmed": False,
            "manual_unit_or_coding_match_confirmed": False,
            "manual_notes": "",
        }

        if usa_df is not None and feature in usa_df.columns:
            u = pd.to_numeric(usa_df[feature], errors="coerce").to_numpy(dtype=float)
            u = u[np.isfinite(u)]
            if u.size > FEATURE_AUDIT_MAX_VALUES_PER_FEATURE:
                step = max(1, int(np.floor(u.size / FEATURE_AUDIT_MAX_VALUES_PER_FEATURE)))
                u = u[::step][:FEATURE_AUDIT_MAX_VALUES_PER_FEATURE]
            row.update({
                "present_in_optional_usa_table": True,
                "usa_n_sampled_values": int(u.size),
                "usa_min": float(np.min(u)) if u.size else np.nan,
                "usa_q01": float(np.quantile(u, 0.01)) if u.size else np.nan,
                "usa_median": float(np.median(u)) if u.size else np.nan,
                "usa_q99": float(np.quantile(u, 0.99)) if u.size else np.nan,
                "usa_max": float(np.max(u)) if u.size else np.nan,
            })
            s_med = row["sweden_median"]
            u_med = row["usa_median"]
            row["usa_to_sweden_median_ratio"] = (
                float(u_med / s_med)
                if np.isfinite(s_med) and np.isfinite(u_med) and abs(s_med) > 1e-12
                else np.nan
            )
        else:
            row["present_in_optional_usa_table"] = False
        rows.append(row)
    return pd.DataFrame(rows)


if SOURCE_DATA_PKL_OVERRIDE is not None:
    SOURCE_DATA_PKL = os.path.abspath(SOURCE_DATA_PKL_OVERRIDE)
    if not os.path.isfile(SOURCE_DATA_PKL):
        raise FileNotFoundError(
            f"SOURCE_DATA_PKL_OVERRIDE does not exist: {SOURCE_DATA_PKL}"
        )
else:
    SOURCE_DATA_PKL = find_dataset_file(
        SOURCE_DATASET_ROOT,
        SOURCE_DATASET_LABEL,
    )

print("SOURCE_DATA_PKL:", SOURCE_DATA_PKL)
print("COMMON_DATA_PKL:", COMMON_DATA_PKL)
print("EXP_ROOT:", EXP_ROOT)

with open(SOURCE_DATA_PKL, "rb") as f:
    source_data = pickle.load(f)

if "X2_by_patient" not in source_data or "y_by_patient" not in source_data:
    raise KeyError("The source PKL must contain X2_by_patient and y_by_patient.")

source_xdict = source_data["X2_by_patient"]
source_y_by_patient = source_data["y_by_patient"]
source_feature_cols = list(source_data.get("feature_cols", []))
if not source_feature_cols:
    raise ValueError("feature_cols is missing from the source PKL.")
if len(source_feature_cols) != len(set(source_feature_cols)):
    duplicates = [x for x, c in Counter(source_feature_cols).items() if c > 1]
    raise ValueError(f"Duplicate source feature names: {duplicates}")

missing_common = [f for f in SELECTED_FEATURES_ORDERED if f not in source_feature_cols]
if missing_common:
    pd.DataFrame({
        "source_feature": source_feature_cols,
    }).to_csv(
        os.path.join(EXP_ROOT, "SOURCE_feature_cols_when_common_features_missing.csv"),
        index=False,
    )
    raise ValueError(
        "The Sweden 120-min PKL does not contain all requested common features. "
        f"Missing: {missing_common}. Inspect the saved source-feature CSV."
    )

selected_idx = [source_feature_cols.index(f) for f in SELECTED_FEATURES_ORDERED]
source_fingerprint = file_fingerprint(SOURCE_DATA_PKL)
common_metadata = {
    "dataset_kind": "Sweden 120-min / 40-min USA-compatible common-feature dataset",
    "source_dataset_fingerprint": source_fingerprint,
    "source_feature_cols": source_feature_cols,
    "selected_feature_cols_ordered": SELECTED_FEATURES_ORDERED,
    "common_dynamic_features": COMMON_DYNAMIC_FEATURES,
    "static_features": STATIC_FEATURES,
    "target_only_features_not_in_input": TARGET_ONLY_FEATURES,
    "excluded_usa_features": EXCLUDED_USA_FEATURES,
    "value_mask_channels": ["value", "mask"],
    "created_by_code_version": CODE_VERSION,
}
common_metadata_hash = stable_hash(common_metadata)
common_metadata["metadata_hash"] = common_metadata_hash

reuse_common = False
if os.path.isfile(COMMON_DATA_PKL) and not REBUILD_COMMON_DATASET:
    with open(COMMON_DATA_PKL, "rb") as f:
        common_data = pickle.load(f)
    existing_meta = common_data.get("common_feature_metadata", {})
    if existing_meta.get("metadata_hash") != common_metadata_hash:
        raise RuntimeError(
            "Existing COMMON_DATA_PKL was built from a different source or feature contract. "
            "Set REBUILD_COMMON_DATASET=True or use a new path."
        )
    reuse_common = True
else:
    common_xdict = {}
    for pid, Xp in source_xdict.items():
        if Xp is None:
            common_xdict[pid] = None
            continue
        arr = np.asarray(Xp)
        if arr.ndim != 4:
            raise ValueError(f"pid={pid}: expected source X ndim=4, got {arr.shape}")
        if arr.shape[-1] != len(source_feature_cols):
            raise ValueError(
                f"pid={pid}: source X feature dimension {arr.shape[-1]} does not match "
                f"len(feature_cols)={len(source_feature_cols)}"
            )
        common_xdict[pid] = np.take(arr, selected_idx, axis=-1).astype(np.float32)

    # Preserve all upstream metadata except the replaced feature tensor/list.
    common_data = {
        key: value
        for key, value in source_data.items()
        if key not in {"X2_by_patient", "feature_cols"}
    }
    common_data["X2_by_patient"] = common_xdict
    common_data["y_by_patient"] = source_y_by_patient
    common_data["feature_cols"] = list(SELECTED_FEATURES_ORDERED)
    common_data["common_feature_metadata"] = common_metadata
    atomic_pickle_dump(common_data, COMMON_DATA_PKL)

print("Common-feature PKL reused:" if reuse_common else "Common-feature PKL created:", COMMON_DATA_PKL)

# Strictly validate the materialized artifact before training.
data = common_data
DATA_PKL = COMMON_DATA_PKL
DATASET_LABEL = "120min_stride40min_overlap67_usa_common17"
X2_by_patient_raw = data["X2_by_patient"]
y_by_patient = data["y_by_patient"]
feature_cols = list(data.get("feature_cols", []))
if feature_cols != SELECTED_FEATURES_ORDERED:
    raise RuntimeError(
        "Common PKL feature order mismatch. "
        f"Expected {SELECTED_FEATURES_ORDERED}, got {feature_cols}"
    )

found_forbidden = [f for f in FORBIDDEN_FEATURES if f in feature_cols]
if STRICT_NO_FORBIDDEN_FEATURES and found_forbidden:
    raise ValueError(f"Forbidden feature leaked into model input: {found_forbidden}")

# Automated audit and manual metadata checklist.
usa_df_optional = load_optional_usa_dataframe(USA_TABULAR_PATH)
feature_audit_df = build_feature_audit(
    source_xdict=source_xdict,
    source_feature_cols=source_feature_cols,
    usa_df=usa_df_optional,
)
for output_root in [COMMON_DATASET_DIR, EXP_ROOT]:
    feature_audit_df.to_csv(
        os.path.join(output_root, "Sweden_USA_common_feature_audit.csv"),
        index=False,
    )
    feature_audit_df[
        [
            "feature",
            "role_in_common_pipeline",
            "expected_unit_or_coding",
            "manual_definition_match_confirmed",
            "manual_unit_or_coding_match_confirmed",
            "manual_notes",
        ]
    ].to_csv(
        os.path.join(output_root, "MANUAL_feature_definition_unit_checklist.csv"),
        index=False,
    )

# Explicit mapping/role table for all 20 USA columns supplied by the user.
mapping_df = pd.DataFrame({
    "usa_feature": USA_FEATURES_REPORTED,
})
mapping_df["sweden_feature"] = mapping_df["usa_feature"]
mapping_df["role"] = mapping_df["usa_feature"].map(expected_role)
mapping_df["included_in_model"] = mapping_df["usa_feature"].isin(SELECTED_FEATURES_ORDERED)
mapping_df["expected_unit_or_coding"] = mapping_df["usa_feature"].map(expected_unit)
mapping_df["reason_if_excluded"] = mapping_df["usa_feature"].map({
    "feats__pna_days": "Used only to construct USA true PMA = GA + PNA/7; including it would leak the target.",
    "feats__apgar_1": "Not part of the pre-specified common static branch.",
    "feats__apgar_5": "Not part of the pre-specified common static branch.",
}).fillna("")
for output_root in [COMMON_DATASET_DIR, EXP_ROOT]:
    mapping_df.to_csv(
        os.path.join(output_root, "Sweden_USA_feature_mapping_and_roles.csv"),
        index=False,
    )

if STOP_AFTER_COMMON_DATASET_AND_AUDIT:
    print("\nAudit-only mode complete.")
    print("Review:", os.path.join(EXP_ROOT, "Sweden_USA_common_feature_audit.csv"))
    print("Review:", os.path.join(EXP_ROOT, "MANUAL_feature_definition_unit_checklist.csv"))
    print("Common PKL:", COMMON_DATA_PKL)
    raise SystemExit(0)

# Separate dynamic CNN input and static branch input.
static_idx = [feature_cols.index(c) for c in STATIC_FEATURES]
dynamic_idx = [feature_cols.index(c) for c in COMMON_DYNAMIC_FEATURES]
static_feature_cols = [feature_cols[i] for i in static_idx]
dynamic_feature_cols = [feature_cols[i] for i in dynamic_idx]

if dynamic_feature_cols != COMMON_DYNAMIC_FEATURES:
    raise RuntimeError("Dynamic feature order is not the declared common order.")
if static_feature_cols != STATIC_FEATURES:
    raise RuntimeError("Static feature order is not the declared common order.")

example_pid = None
for pid, Xp in X2_by_patient_raw.items():
    if Xp is not None and hasattr(Xp, "shape") and Xp.shape[0] > 0:
        example_pid = pid
        break
if example_pid is None:
    raise ValueError("No valid patient found in X2_by_patient_raw.")

example_shape = np.asarray(X2_by_patient_raw[example_pid]).shape
if len(example_shape) != 4:
    raise ValueError(f"Expected X shape (n_windows, 2, T, F), got {example_shape}")
if example_shape[1] != 2:
    raise ValueError(f"Expected value+mask channels, got shape {example_shape}")
if example_shape[-1] != len(feature_cols):
    raise ValueError("feature_cols length does not match common X feature dimension.")

N_TIME = int(example_shape[2])
N_DYNAMIC_FEATURES = len(dynamic_feature_cols)
N_STATIC = len(static_feature_cols)

print("\nCommon feature structure:")
print("Example pid:", example_pid)
print("Example X shape before static separation:", example_shape)
print("Dynamic features:", dynamic_feature_cols)
print("Static features:", static_feature_cols)
print("N dynamic features:", N_DYNAMIC_FEATURES)
print("N static features:", N_STATIC)
print("Forbidden model-input features found:", found_forbidden)

X2_by_patient = {}
Xstatic_by_patient = {}
for pid, Xp in X2_by_patient_raw.items():
    if Xp is None:
        X2_by_patient[pid] = None
        Xstatic_by_patient[pid] = None
        continue

    Xp = np.asarray(Xp)
    if Xp.ndim != 4:
        raise ValueError(f"pid={pid}: expected X ndim=4, got shape={Xp.shape}")
    if Xp.shape[1] != 2:
        raise ValueError(f"pid={pid}: expected 2 channels, got shape={Xp.shape}")
    if Xp.shape[2] != N_TIME:
        raise ValueError(
            f"pid={pid}: time dimension mismatch, expected {N_TIME}, got {Xp.shape[2]}"
        )
    if Xp.shape[-1] != len(feature_cols):
        raise ValueError(f"pid={pid}: common feature dimension mismatch")
    if not np.isfinite(Xp).all():
        raise ValueError(
            f"pid={pid}: non-finite values remain in imputed common X "
            f"({int((~np.isfinite(Xp)).sum())} values)"
        )

    if Xp.shape[0] == 0:
        X2_by_patient[pid] = np.empty(
            (0, 2, Xp.shape[2], N_DYNAMIC_FEATURES), dtype=np.float32
        )
        Xstatic_by_patient[pid] = np.empty((0, N_STATIC), dtype=np.float32)
    else:
        static_all = np.take(Xp[:, 0, :, :], static_idx, axis=-1)
        static_reference = np.take(Xp[0, 0, 0, :], static_idx, axis=-1)
        expected_static_shape = (Xp.shape[0], Xp.shape[2], N_STATIC)
        if static_all.shape != expected_static_shape:
            raise RuntimeError(
                f"pid={pid}: static indexing produced {static_all.shape}, "
                f"expected {expected_static_shape}"
            )
        if static_reference.shape != (N_STATIC,):
            raise RuntimeError(
                f"pid={pid}: static reference shape {static_reference.shape}, "
                f"expected {(N_STATIC,)}"
            )
        if not np.allclose(
            static_all,
            static_reference.reshape(1, 1, -1),
            rtol=0.0,
            atol=1e-6,
            equal_nan=False,
        ):
            raise ValueError(
                f"pid={pid}: one or more static common features vary across "
                "windows/time; refusing to silently take frame 0."
            )

        X2_by_patient[pid] = np.take(Xp, dynamic_idx, axis=-1).astype(np.float32)
        Xstatic_by_patient[pid] = np.take(
            Xp[:, 0, 0, :], static_idx, axis=-1
        ).astype(np.float32)

pd.DataFrame({"dynamic_feature": dynamic_feature_cols}).to_csv(
    os.path.join(EXP_ROOT, "dynamic_features_for_cnn.csv"), index=False
)
pd.DataFrame({"static_feature": static_feature_cols}).to_csv(
    os.path.join(EXP_ROOT, "static_features_for_branch.csv"), index=False
)

# ============================================================
# 3. Data utilities
# ============================================================
def patient_invalid_reason(pid):
    if pid not in X2_by_patient:
        return "missing_X"
    if pid not in Xstatic_by_patient:
        return "missing_static"
    if pid not in y_by_patient:
        return "missing_target"

    Xp = X2_by_patient[pid]
    Sp = Xstatic_by_patient[pid]
    yp = y_by_patient[pid]

    if Xp is None:
        return "X_is_none"
    if Sp is None:
        return "static_is_none"
    if yp is None:
        return "target_is_none"

    yp = np.asarray(yp)
    if Xp.ndim != 4 or Xp.shape[1:] != (2, N_TIME, N_DYNAMIC_FEATURES):
        return f"invalid_dynamic_shape_{tuple(Xp.shape)}"
    if Sp.ndim != 2 or Sp.shape[1] != N_STATIC:
        return f"invalid_static_shape_{tuple(Sp.shape)}"
    if Xp.shape[0] == 0 or Sp.shape[0] == 0 or len(yp) == 0:
        return "zero_windows"
    if Xp.shape[0] != Sp.shape[0] or Xp.shape[0] != len(yp):
        return (
            f"length_mismatch_X{Xp.shape[0]}_"
            f"S{Sp.shape[0]}_y{len(yp)}"
        )
    if not np.isfinite(Xp).all():
        return "nonfinite_dynamic_input"
    if not np.isfinite(Sp).all():
        return "nonfinite_static_input"
    if not np.isfinite(yp.astype(float)).all():
        return "nonfinite_target"
    return None


def is_valid_patient(pid) -> bool:
    return patient_invalid_reason(pid) is None


def flatten_windows(pid_list):
    X_list, S_list, y_list, pid_per_window = [], [], [], []
    skipped = 0

    for pid in pid_list:
        reason = patient_invalid_reason(pid)
        if reason is not None:
            skipped += 1
            continue

        n_windows = int(X2_by_patient[pid].shape[0])
        X_list.append(X2_by_patient[pid])
        S_list.append(Xstatic_by_patient[pid])
        y_list.append(np.asarray(y_by_patient[pid], dtype=np.float32))
        pid_per_window.extend([pid] * n_windows)

    if not X_list:
        X = np.empty(
            (0, 2, N_TIME, N_DYNAMIC_FEATURES),
            dtype=np.float32,
        )
        S = np.empty((0, N_STATIC), dtype=np.float32)
        y = np.empty((0,), dtype=np.float32)
        window_pids = np.empty((0,), dtype=object)
    else:
        X = np.concatenate(X_list, axis=0).astype(np.float32)
        S = np.concatenate(S_list, axis=0).astype(np.float32)
        y = np.concatenate(y_list, axis=0).astype(np.float32)
        window_pids = np.asarray(pid_per_window, dtype=object)

    return X, S, y, window_pids, skipped


def patient_balanced_window_weights(window_pids):
    window_pids = np.asarray(window_pids, dtype=object)
    if len(window_pids) == 0:
        return np.empty((0,), dtype=np.float64)

    counts = Counter(window_pids.tolist())
    weights = np.asarray(
        [1.0 / counts[pid] for pid in window_pids],
        dtype=np.float64,
    )

    patient_total_weights = {
        pid: float(weights[window_pids == pid].sum())
        for pid in counts
    }
    if not np.allclose(
        list(patient_total_weights.values()),
        1.0,
        rtol=0.0,
        atol=1e-10,
    ):
        raise RuntimeError(
            "Patient-balanced weights failed the equal-total-weight check."
        )
    return weights


def compute_norm_stats(
    X_train_raw,
    S_train_raw,
    train_window_pids,
):
    """
    Patient-balanced, feature-wise z-score normalization.

    Each patient contributes equal total weight regardless of how many
    windows they have. For every dynamic feature, statistics are calculated
    over training windows/time points only. The mask channel is unchanged.
    """
    vals = X_train_raw[:, 0, :, :].astype(np.float64, copy=False)

    if not np.isfinite(vals).all():
        bad = int((~np.isfinite(vals)).sum())
        raise ValueError(
            f"Non-finite training dynamic values: {bad}"
        )
    if not np.isfinite(S_train_raw).all():
        bad = int((~np.isfinite(S_train_raw)).sum())
        raise ValueError(
            f"Non-finite training static values: {bad}"
        )

    if PATIENT_BALANCED_NORMALIZATION:
        window_weights = patient_balanced_window_weights(
            train_window_pids
        )
    else:
        window_weights = np.ones(
            len(X_train_raw),
            dtype=np.float64,
        )

    weight_sum = float(window_weights.sum())
    if weight_sum <= 0:
        raise ValueError("Normalization weights sum to zero.")

    # Average time within each window first, then apply the per-window
    # patient-balancing weight. This is equivalent to applying the same
    # window weight to every frame.
    per_window_feature_mean = vals.mean(axis=1)  # (N, F)
    mean_f = np.sum(
        per_window_feature_mean * window_weights[:, None],
        axis=0,
    ) / weight_sum

    centered_sq_time_mean = (
        (vals - mean_f.reshape(1, 1, -1)) ** 2
    ).mean(axis=1)
    var_f = np.sum(
        centered_sq_time_mean * window_weights[:, None],
        axis=0,
    ) / weight_sum

    mean0 = mean_f.reshape(1, 1, -1).astype(np.float32)
    std0 = np.sqrt(np.maximum(var_f, 1e-12)).reshape(
        1, 1, -1
    ).astype(np.float32)
    std0 = np.maximum(std0, 1e-6).astype(np.float32)

    expected_shape = (1, 1, vals.shape[-1])
    if mean0.shape != expected_shape or std0.shape != expected_shape:
        raise RuntimeError(
            "Dynamic normalization is not per-feature: "
            f"mean={mean0.shape}, std={std0.shape}, "
            f"expected={expected_shape}"
        )

    static64 = S_train_raw.astype(np.float64, copy=False)
    static_mean64 = np.sum(
        static64 * window_weights[:, None],
        axis=0,
    ) / weight_sum
    static_var64 = np.sum(
        ((static64 - static_mean64) ** 2)
        * window_weights[:, None],
        axis=0,
    ) / weight_sum

    static_mean = static_mean64.astype(np.float32)
    static_std = np.sqrt(
        np.maximum(static_var64, 1e-12)
    ).astype(np.float32)
    static_std = np.maximum(static_std, 1e-6).astype(np.float32)

    return mean0, std0, static_mean, static_std


def apply_norm(X, mean0, std0):
    X = X.copy()
    if not np.isfinite(X).all():
        bad = int((~np.isfinite(X)).sum())
        raise ValueError(
            f"Non-finite values before dynamic normalization: {bad}"
        )

    X[:, 0, :, :] = (
        X[:, 0, :, :] - mean0
    ) / std0
    # Mask channel X[:, 1, :, :] intentionally remains unchanged.
    return X.astype(np.float32)


def apply_static_norm(S, static_mean, static_std):
    if not np.isfinite(S).all():
        bad = int((~np.isfinite(S)).sum())
        raise ValueError(
            f"Non-finite values before static normalization: {bad}"
        )
    return (
        (S - static_mean) / static_std
    ).astype(np.float32)


class WindowDataset(Dataset):
    def __init__(self, X, S, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.S = torch.tensor(S, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.S[idx], self.y[idx]


def make_loader_from_dataset(
    dataset,
    batch_size,
    shuffle,
    seed=None,
    sample_weights_tensor=None,
):
    """
    Build a DataLoader from an already-created Dataset.

    For patient-balanced training, sample_weights_tensor is computed once
    outside the epoch loop. Each epoch creates only a new generator,
    WeightedRandomSampler and DataLoader; the full X/S/y tensors are not
    copied again.
    """
    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(int(seed))

    sampler = None
    if sample_weights_tensor is not None:
        if len(sample_weights_tensor) != len(dataset):
            raise ValueError(
                "sample_weights_tensor length does not match dataset."
            )
        sampler = WeightedRandomSampler(
            weights=sample_weights_tensor,
            num_samples=len(sample_weights_tensor),
            replacement=True,
            generator=generator,
        )
        shuffle = False

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        generator=generator if sampler is None else None,
        num_workers=N_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def make_loader(
    X,
    S,
    y,
    batch_size,
    shuffle,
    seed=None,
):
    """
    Convenience wrapper for loaders that are created only once, such as
    validation and test loaders.
    """
    dataset = WindowDataset(X, S, y)
    return make_loader_from_dataset(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
        sample_weights_tensor=None,
    )


def prepare_training_dataset_and_weights(
    X,
    S,
    y,
    window_pids,
):
    """
    Create the training Dataset and patient-balanced weights exactly once.

    Returns
    -------
    train_dataset : WindowDataset
        Owns one Torch copy of X, S and y for the whole model fit.
    sample_weights_tensor : torch.DoubleTensor or None
        Reused by every epoch's WeightedRandomSampler.
    """
    train_dataset = WindowDataset(X, S, y)

    if PATIENT_BALANCED_TRAINING:
        if window_pids is None:
            raise ValueError(
                "window_pids are required for patient-balanced training."
            )
        sample_weights = patient_balanced_window_weights(
            window_pids
        )
        sample_weights_tensor = torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        )
    else:
        sample_weights_tensor = None

    return train_dataset, sample_weights_tensor

# ============================================================
# 4. Model
# ============================================================
class CNN2DReg(nn.Module):
    def __init__(
        self,
        architecture: str,
        n_static: int,
        static_dim: int = 16,
        dropout: float = 0.2,
        head_dim: int = 32,
    ):
        super().__init__()
        if architecture not in ARCH_CONFIGS:
            raise ValueError(f"Unknown architecture: {architecture}")

        cfg = ARCH_CONFIGS[architecture]
        channels = cfg["channels"]
        use_mid_pool = cfg["use_mid_pool"]
        head_type = cfg["head_type"]

        self.architecture = architecture
        self.channels = channels
        self.use_cnn = len(channels) > 0
        self.head_type = head_type

        if self.use_cnn:
            blocks = []
            cin = 2
            for cout in channels:
                layers = [
                    nn.Conv2d(cin, cout, kernel_size=(3, 3), padding=(1, 1), bias=False),
                    nn.BatchNorm2d(cout),
                    nn.ReLU(inplace=True),
                ]
                if use_mid_pool:
                    layers.append(nn.MaxPool2d(kernel_size=(2, 1)))
                blocks.append(nn.Sequential(*layers))
                cin = cout
            self.features = nn.Sequential(*blocks)
            self.gap = nn.AdaptiveAvgPool2d((1, 1))
            cnn_dim = channels[-1]
        else:
            self.features = None
            self.gap = None
            cnn_dim = 0

        self.flatten = nn.Flatten()
        self.static_branch = nn.Sequential(
            nn.Linear(n_static, static_dim),
            nn.ReLU(inplace=True),
        )

        combined_dim = cnn_dim + static_dim
        if head_type == "simple":
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(combined_dim, 1),
            )
        elif head_type == "hidden":
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(combined_dim, head_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(head_dim, 1),
            )
        else:
            raise ValueError(f"Unknown head_type: {head_type}")

    def forward(self, x, s):
        pieces = []
        if self.use_cnn:
            x = self.features(x)
            x = self.gap(x)
            x = self.flatten(x)
            pieces.append(x)
        s = self.static_branch(s)
        pieces.append(s)
        z = torch.cat(pieces, dim=1) if len(pieces) > 1 else pieces[0]
        return self.head(z).squeeze(1)


class EarlyStopping:
    def __init__(self, patience=20, min_delta=5e-4, mode="min"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return True
        if self.mode == "min":
            improved = current_score < (self.best_score - self.min_delta)
        else:
            improved = current_score > (self.best_score + self.min_delta)
        if improved:
            self.best_score = current_score
            self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False

# ============================================================
# 5. Evaluation utilities
# ============================================================
@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    model.eval()
    loss_sum, n = 0.0, 0
    for x, s, y in loader:
        x, s, y = x.to(DEVICE), s.to(DEVICE), y.to(DEVICE)
        pred = model(x, s)
        loss = criterion(pred, y)
        loss_sum += loss.item() * y.numel()
        n += y.numel()
    return loss_sum / max(n, 1)


@torch.no_grad()
def evaluate_window_level(model, loader):
    model.eval()
    mae_sum, se_sum, n = 0.0, 0.0, 0
    for x, s, y in loader:
        x, s, y = x.to(DEVICE), s.to(DEVICE), y.to(DEVICE)
        pred = model(x, s)
        err = pred - y
        mae_sum += torch.abs(err).sum().item()
        se_sum += (err ** 2).sum().item()
        n += y.numel()
    return mae_sum / max(n, 1), float(np.sqrt(se_sum / max(n, 1)))


def patient_regression_metrics(df, pred_col="pred_patient", true_col="true_patient"):
    if len(df) == 0:
        return {"mae": np.nan, "rmse": np.nan, "n_patients": 0}
    err = df[pred_col].values - df[true_col].values
    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "n_patients": int(len(df)),
    }


@torch.no_grad()
def predict_windows_batched(
    model,
    X,
    S,
    batch_size=PRED_BATCH_SIZE,
):
    model.eval()
    predictions = []

    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))
        x = torch.as_tensor(
            X[start:end],
            dtype=torch.float32,
            device=DEVICE,
        )
        s = torch.as_tensor(
            S[start:end],
            dtype=torch.float32,
            device=DEVICE,
        )
        pred = model(x, s)
        predictions.append(
            pred.detach().cpu().numpy().astype(np.float32)
        )

    if not predictions:
        return np.empty((0,), dtype=np.float32)
    return np.concatenate(predictions, axis=0)


@torch.no_grad()
def predict_patient_level(
    model,
    pid_list,
    mean0,
    std0,
    static_mean,
    static_std,
    meta=None,
):
    model.eval()
    rows = []
    meta = meta or {}

    for pid in pid_list:
        if not is_valid_patient(pid):
            continue

        Xp = apply_norm(
            X2_by_patient[pid].astype(
                np.float32,
                copy=True,
            ),
            mean0,
            std0,
        )
        Sp = apply_static_norm(
            Xstatic_by_patient[pid].astype(
                np.float32,
                copy=True,
            ),
            static_mean,
            static_std,
        )
        yp = np.asarray(
            y_by_patient[pid],
            dtype=np.float32,
        )

        pred_w = predict_windows_batched(model, Xp, Sp)

        row = {
            "pid": pid,
            "pred_patient": float(pred_w.mean()),
            "true_patient": float(yp.mean()),
            "pred_window_mean": float(pred_w.mean()),
            "true_window_mean": float(yp.mean()),
            "pred_window_std": float(pred_w.std(ddof=0)),
            "true_window_std": float(yp.std(ddof=0)),
            "n_windows": int(len(yp)),
        }
        row.update(meta)
        rows.append(row)

    return pd.DataFrame(rows)


@torch.no_grad()
def collect_window_predictions(
    model,
    pid_list,
    mean0,
    std0,
    static_mean,
    static_std,
    meta=None,
):
    model.eval()
    rows = []
    meta = meta or {}

    for pid in pid_list:
        if not is_valid_patient(pid):
            continue

        Xp = apply_norm(
            X2_by_patient[pid].astype(
                np.float32,
                copy=True,
            ),
            mean0,
            std0,
        )
        Sp = apply_static_norm(
            Xstatic_by_patient[pid].astype(
                np.float32,
                copy=True,
            ),
            static_mean,
            static_std,
        )
        yp = np.asarray(
            y_by_patient[pid],
            dtype=np.float32,
        )

        pred_w = predict_windows_batched(model, Xp, Sp)

        for i in range(len(yp)):
            row = {
                "pid": pid,
                "window_idx": i,
                "window_id": f"{pid}_{i}",
                "true_window": float(yp[i]),
                "pred_window": float(pred_w[i]),
                "error_window": float(pred_w[i] - yp[i]),
            }
            row.update(meta)
            rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 6. Training functions
# ============================================================
def make_criterion(beta):
    try:
        return nn.SmoothL1Loss(beta=beta)
    except TypeError as exc:
        raise RuntimeError(
            "This code requires a PyTorch version whose "
            "SmoothL1Loss supports beta."
        ) from exc


def build_optimizer(model, params):
    if params["optimizer"] == "Adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=params["lr"],
            weight_decay=params["weight_decay"],
        )
    if params["optimizer"] == "AdamW":
        return torch.optim.AdamW(
            model.parameters(),
            lr=params["lr"],
            weight_decay=params["weight_decay"],
        )
    raise ValueError(
        f"Unknown optimizer: {params['optimizer']}"
    )


def build_scheduler(optimizer, params):
    scheduler_type = params.get("scheduler_type", "none")
    if scheduler_type == "none":
        return None
    if scheduler_type == "cosine":
        eta_min = float(
            params["lr"]
            * params.get("cosine_eta_min_ratio", 0.01)
        )
        # T_max remains MAX_EPOCHS in both inner training and full refit.
        # Therefore the LR trajectory at epoch e is identical in both.
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=MAX_EPOCHS,
            eta_min=eta_min,
        )
    raise ValueError(
        f"Unknown scheduler_type: {scheduler_type}"
    )


def train_one_model(
    run_name,
    train_pids,
    val_pids,
    test_pids=None,
    params=None,
    max_epochs=MAX_EPOCHS,
    save_checkpoint_path=None,
    evaluate_test=False,
    verbose=False,
    run_seed=42,
    meta=None,
):
    seed_everything(run_seed)
    params = params or {}
    meta = meta or {}

    (
        X_train_raw,
        S_train_raw,
        y_train,
        train_window_pids,
        skipped_train,
    ) = flatten_windows(train_pids)
    (
        X_val_raw,
        S_val_raw,
        y_val,
        _,
        skipped_val,
    ) = flatten_windows(val_pids)

    if X_train_raw.shape[0] == 0:
        raise ValueError(f"{run_name}: empty training windows")
    if X_val_raw.shape[0] == 0:
        raise ValueError(f"{run_name}: empty validation windows")

    mean0, std0, static_mean, static_std = compute_norm_stats(
        X_train_raw,
        S_train_raw,
        train_window_pids,
    )
    X_train = apply_norm(X_train_raw, mean0, std0)
    S_train = apply_static_norm(
        S_train_raw,
        static_mean,
        static_std,
    )
    X_val = apply_norm(X_val_raw, mean0, std0)
    S_val = apply_static_norm(
        S_val_raw,
        static_mean,
        static_std,
    )

    train_dataset, train_sample_weights = (
        prepare_training_dataset_and_weights(
            X_train,
            S_train,
            y_train,
            train_window_pids,
        )
    )

    val_loader = make_loader(
        X_val,
        S_val,
        y_val,
        batch_size=256,
        shuffle=False,
    )

    model = CNN2DReg(
        architecture=params["architecture"],
        n_static=N_STATIC,
        static_dim=params["static_dim"],
        dropout=params["dropout"],
        head_dim=params.get("head_dim", 32),
    ).to(DEVICE)
    n_params = count_parameters(model)

    criterion = make_criterion(
        params.get("smooth_l1_beta", 1.0)
    )
    optimizer = build_optimizer(model, params)
    scheduler = build_scheduler(optimizer, params)

    early_stopper = EarlyStopping(
        patience=params.get("patience", 20),
        min_delta=params.get("min_delta", 5e-4),
        mode="min",
    )

    best_state = None
    best_epoch = -1
    best_metric = float("inf")
    history = []

    for epoch in range(1, max_epochs + 1):
        # Epoch-specific deterministic sampling makes trial comparisons fair.
        # The Dataset and all X/S/y tensors were created once above.
        train_loader = make_loader_from_dataset(
            dataset=train_dataset,
            batch_size=params["batch_size"],
            shuffle=not PATIENT_BALANCED_TRAINING,
            seed=run_seed + epoch,
            sample_weights_tensor=train_sample_weights,
        )

        model.train()
        loss_sum, n = 0.0, 0

        for x, s, y in train_loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x, s)
            loss = criterion(pred, y)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"{run_name}: non-finite training loss "
                    f"at epoch {epoch}"
                )

            loss.backward()
            grad_clip = params.get("grad_clip_norm", 0.0)
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip,
                )
            optimizer.step()

            loss_sum += loss.item() * y.numel()
            n += y.numel()

        train_loss = loss_sum / max(n, 1)
        val_loss = evaluate_loss(
            model,
            val_loader,
            criterion,
        )
        val_win_mae, val_win_rmse = evaluate_window_level(
            model,
            val_loader,
        )
        val_pat_df = predict_patient_level(
            model,
            val_pids,
            mean0,
            std0,
            static_mean,
            static_std,
            meta={
                **meta,
                "dataset": "val",
                "run_name": run_name,
            },
        )
        val_pat_metrics = patient_regression_metrics(
            val_pat_df
        )
        current_metric = val_pat_metrics["mae"]

        # Cosine scheduler is epoch-based and independent of val/test metrics.
        if scheduler is not None:
            scheduler.step()

        improved = early_stopper.step(current_metric)
        if improved:
            best_metric = current_metric
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            if save_checkpoint_path is not None:
                atomic_torch_save(
                    {
                        "model_state": best_state,
                        "params": params,
                        "n_params": n_params,
                        "best_epoch": best_epoch,
                        "best_val_patient_mae": best_metric,
                        "mean0": mean0,
                        "std0": std0,
                        "static_mean": static_mean,
                        "static_std": static_std,
                        "dynamic_feature_cols": dynamic_feature_cols,
                        "static_feature_cols": static_feature_cols,
                        "run_name": run_name,
                        "meta": meta,
                    },
                    save_checkpoint_path,
                )

        history.append({
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_window_mae": float(val_win_mae),
            "val_window_rmse": float(val_win_rmse),
            "val_patient_mae": float(
                val_pat_metrics["mae"]
            ),
            "val_patient_rmse": float(
                val_pat_metrics["rmse"]
            ),
            "lr": float(
                optimizer.param_groups[0]["lr"]
            ),
        })

        if verbose:
            print(
                f"[{run_name}] epoch={epoch:03d} "
                f"train_loss={train_loss:.4f} "
                f"val_pat_MAE={val_pat_metrics['mae']:.4f} "
                f"val_win_MAE={val_win_mae:.4f}"
            )

        if early_stopper.should_stop:
            break

    if best_state is None:
        raise RuntimeError(
            f"{run_name}: no best_state saved"
        )

    model.load_state_dict(best_state)
    model.eval()

    val_win_mae, val_win_rmse = evaluate_window_level(
        model,
        val_loader,
    )
    val_pat_df = predict_patient_level(
        model,
        val_pids,
        mean0,
        std0,
        static_mean,
        static_std,
        meta={
            **meta,
            "dataset": "val",
            "run_name": run_name,
        },
    )
    val_pat_metrics = patient_regression_metrics(
        val_pat_df
    )
    val_window_df = collect_window_predictions(
        model,
        val_pids,
        mean0,
        std0,
        static_mean,
        static_std,
        meta={
            **meta,
            "dataset": "val",
            "run_name": run_name,
        },
    )

    test_pat_df = pd.DataFrame()
    test_window_df = pd.DataFrame()
    test_win_mae = np.nan
    test_win_rmse = np.nan
    test_pat_metrics = {
        "mae": np.nan,
        "rmse": np.nan,
        "n_patients": 0,
    }

    if evaluate_test:
        if test_pids is None:
            raise ValueError(
                "test_pids is required when evaluate_test=True"
            )
        (
            X_test_raw,
            S_test_raw,
            y_test,
            _,
            skipped_test,
        ) = flatten_windows(test_pids)
        X_test = apply_norm(
            X_test_raw,
            mean0,
            std0,
        )
        S_test = apply_static_norm(
            S_test_raw,
            static_mean,
            static_std,
        )
        test_loader = make_loader(
            X_test,
            S_test,
            y_test,
            batch_size=256,
            shuffle=False,
        )
        test_win_mae, test_win_rmse = evaluate_window_level(
            model,
            test_loader,
        )
        test_pat_df = predict_patient_level(
            model,
            test_pids,
            mean0,
            std0,
            static_mean,
            static_std,
            meta={
                **meta,
                "dataset": "test",
                "run_name": run_name,
            },
        )
        test_pat_metrics = patient_regression_metrics(
            test_pat_df
        )
        test_window_df = collect_window_predictions(
            model,
            test_pids,
            mean0,
            std0,
            static_mean,
            static_std,
            meta={
                **meta,
                "dataset": "test",
                "run_name": run_name,
            },
        )
    else:
        skipped_test = 0

    return {
        "model": model,
        "params": params,
        "n_params": int(n_params),
        "best_epoch": int(best_epoch),
        "best_val_patient_mae": float(best_metric),
        "history": pd.DataFrame(history),
        "val_patient_df": val_pat_df,
        "val_window_df": val_window_df,
        "test_patient_df": test_pat_df,
        "test_window_df": test_window_df,
        "val_patient_mae": float(
            val_pat_metrics["mae"]
        ),
        "val_patient_rmse": float(
            val_pat_metrics["rmse"]
        ),
        "val_window_mae": float(val_win_mae),
        "val_window_rmse": float(val_win_rmse),
        "test_patient_mae": float(
            test_pat_metrics["mae"]
        ),
        "test_patient_rmse": float(
            test_pat_metrics["rmse"]
        ),
        "test_window_mae": float(test_win_mae),
        "test_window_rmse": float(test_win_rmse),
        "skipped_train": int(skipped_train),
        "skipped_val": int(skipped_val),
        "skipped_test": int(skipped_test),
    }


def train_full_outer_model(
    run_name,
    train_pids,
    test_pids,
    params,
    n_epochs,
    save_checkpoint_path=None,
    resume_checkpoint_path=None,
    verbose=False,
    run_seed=42,
    meta=None,
    experiment_hash=None,
):
    """
    Train on all outer-train patients with fixed epochs selected by inner CV.

    The progress checkpoint contains model, optimizer, scheduler, history,
    normalization statistics and RNG states. Restarting the script resumes
    from the next epoch rather than restarting this seed.
    """
    seed_everything(run_seed)
    meta = meta or {}
    n_epochs = int(
        max(1, min(int(n_epochs), MAX_EPOCHS))
    )

    (
        X_train_raw,
        S_train_raw,
        y_train,
        train_window_pids,
        skipped_train,
    ) = flatten_windows(train_pids)
    (
        X_test_raw,
        S_test_raw,
        y_test,
        _,
        skipped_test,
    ) = flatten_windows(test_pids)

    if X_train_raw.shape[0] == 0:
        raise ValueError(
            f"{run_name}: empty full outer-training windows"
        )
    if X_test_raw.shape[0] == 0:
        raise ValueError(
            f"{run_name}: empty outer-test windows"
        )

    mean0, std0, static_mean, static_std = compute_norm_stats(
        X_train_raw,
        S_train_raw,
        train_window_pids,
    )

    X_train = apply_norm(
        X_train_raw,
        mean0,
        std0,
    )
    S_train = apply_static_norm(
        S_train_raw,
        static_mean,
        static_std,
    )
    X_test = apply_norm(
        X_test_raw,
        mean0,
        std0,
    )
    S_test = apply_static_norm(
        S_test_raw,
        static_mean,
        static_std,
    )

    train_dataset, train_sample_weights = (
        prepare_training_dataset_and_weights(
            X_train,
            S_train,
            y_train,
            train_window_pids,
        )
    )

    test_loader = make_loader(
        X_test,
        S_test,
        y_test,
        batch_size=256,
        shuffle=False,
    )

    model = CNN2DReg(
        architecture=params["architecture"],
        n_static=N_STATIC,
        static_dim=params["static_dim"],
        dropout=params["dropout"],
        head_dim=params.get("head_dim", 32),
    ).to(DEVICE)
    n_params = count_parameters(model)
    criterion = make_criterion(
        params.get("smooth_l1_beta", 1.0)
    )
    optimizer = build_optimizer(model, params)
    scheduler = build_scheduler(optimizer, params)

    training_identity = {
        "experiment_hash": experiment_hash,
        "run_name": run_name,
        "run_seed": int(run_seed),
        "n_epochs": int(n_epochs),
        "params_hash": stable_hash(params),
        "train_pids_hash": stable_hash(
            [str(x) for x in train_pids]
        ),
        "test_pids_hash": stable_hash(
            [str(x) for x in test_pids]
        ),
    }

    history = []
    start_epoch = 1
    loaded_complete_checkpoint = False

    # A complete final checkpoint can be reused for evaluation if the
    # process stopped after training but before CSV/marker creation.
    if (
        save_checkpoint_path is not None
        and os.path.isfile(save_checkpoint_path)
    ):
        completed = safe_torch_load(
            save_checkpoint_path,
            map_location=DEVICE,
        )
        if completed.get("training_identity") != training_identity:
            raise RuntimeError(
                f"{run_name}: final checkpoint identity mismatch."
            )
        if completed.get("training_complete", False):
            model.load_state_dict(completed["model_state"])
            history = completed.get("history", [])
            loaded_complete_checkpoint = True
            start_epoch = n_epochs + 1

    if (
        not loaded_complete_checkpoint
        and resume_checkpoint_path is not None
        and os.path.isfile(resume_checkpoint_path)
    ):
        progress = safe_torch_load(
            resume_checkpoint_path,
            map_location=DEVICE,
        )
        if progress.get("training_identity") != training_identity:
            raise RuntimeError(
                f"{run_name}: resume checkpoint identity mismatch."
            )

        model.load_state_dict(progress["model_state"])
        optimizer.load_state_dict(progress["optimizer_state"])
        if (
            scheduler is not None
            and progress.get("scheduler_state") is not None
        ):
            scheduler.load_state_dict(
                progress["scheduler_state"]
            )

        history = progress.get("history", [])
        start_epoch = int(progress["epoch"]) + 1
        restore_rng_state(progress.get("rng_state"))

        print(
            f"[{run_name}] resuming from epoch "
            f"{start_epoch}/{n_epochs}"
        )

    for epoch in range(start_epoch, n_epochs + 1):
        # Reuse the same Dataset and weight tensor after an epoch-level resume.
        train_loader = make_loader_from_dataset(
            dataset=train_dataset,
            batch_size=params["batch_size"],
            shuffle=not PATIENT_BALANCED_TRAINING,
            seed=run_seed + epoch,
            sample_weights_tensor=train_sample_weights,
        )

        model.train()
        loss_sum, n = 0.0, 0

        for x, s, y in train_loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x, s)
            loss = criterion(pred, y)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"{run_name}: non-finite training loss "
                    f"at epoch {epoch}"
                )

            loss.backward()
            grad_clip = params.get("grad_clip_norm", 0.0)
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip,
                )
            optimizer.step()

            loss_sum += loss.item() * y.numel()
            n += y.numel()

        train_loss = loss_sum / max(n, 1)
        if scheduler is not None:
            scheduler.step()

        history.append({
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "lr": float(
                optimizer.param_groups[0]["lr"]
            ),
        })

        if (
            CHECKPOINT_EVERY_EPOCH
            and resume_checkpoint_path is not None
        ):
            atomic_torch_save(
                {
                    "training_identity": training_identity,
                    "epoch": int(epoch),
                    "model_state": {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    },
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": (
                        scheduler.state_dict()
                        if scheduler is not None
                        else None
                    ),
                    "history": history,
                    "rng_state": capture_rng_state(),
                    "mean0": mean0,
                    "std0": std0,
                    "static_mean": static_mean,
                    "static_std": static_std,
                },
                resume_checkpoint_path,
            )

        if verbose and (
            epoch == 1
            or epoch == n_epochs
            or epoch % 10 == 0
        ):
            print(
                f"[{run_name}] epoch={epoch:03d}/"
                f"{n_epochs:03d} "
                f"train_loss={train_loss:.4f} "
                f"lr={optimizer.param_groups[0]['lr']:.3e}"
            )

    model.eval()

    test_win_mae, test_win_rmse = evaluate_window_level(
        model,
        test_loader,
    )
    test_pat_df = predict_patient_level(
        model,
        test_pids,
        mean0,
        std0,
        static_mean,
        static_std,
        meta={
            **meta,
            "dataset": "outer_test",
            "run_name": run_name,
        },
    )
    test_pat_metrics = patient_regression_metrics(
        test_pat_df
    )
    test_window_df = collect_window_predictions(
        model,
        test_pids,
        mean0,
        std0,
        static_mean,
        static_std,
        meta={
            **meta,
            "dataset": "outer_test",
            "run_name": run_name,
        },
    )

    if save_checkpoint_path is not None:
        atomic_torch_save(
            {
                "training_identity": training_identity,
                "training_complete": True,
                "model_state": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                "params": params,
                "n_params": int(n_params),
                "n_epochs": int(n_epochs),
                "history": history,
                "training_mode": (
                    "full_outer_train_fixed_epoch_"
                    "patient_balanced"
                ),
                "mean0": mean0,
                "std0": std0,
                "static_mean": static_mean,
                "static_std": static_std,
                "dynamic_feature_cols": dynamic_feature_cols,
                "static_feature_cols": static_feature_cols,
                "run_name": run_name,
                "run_seed": int(run_seed),
                "meta": meta,
            },
            save_checkpoint_path,
        )

    if (
        resume_checkpoint_path is not None
        and os.path.isfile(resume_checkpoint_path)
    ):
        os.remove(resume_checkpoint_path)

    return {
        "model": model,
        "params": params,
        "n_params": int(n_params),
        "n_epochs": int(n_epochs),
        "history": pd.DataFrame(history),
        "test_patient_df": test_pat_df,
        "test_window_df": test_window_df,
        "test_patient_mae": float(
            test_pat_metrics["mae"]
        ),
        "test_patient_rmse": float(
            test_pat_metrics["rmse"]
        ),
        "test_window_mae": float(test_win_mae),
        "test_window_rmse": float(test_win_rmse),
        "skipped_train": int(skipped_train),
        "skipped_test": int(skipped_test),
    }

# ============================================================
# 7. Stratification utilities
# ============================================================
def make_strat_bins(y_values, n_splits=5, max_bins=10):
    y_values = np.asarray(y_values, dtype=float)
    if not np.isfinite(y_values).all():
        raise ValueError(
            "Non-finite patient targets passed to stratification."
        )

    for n_bins in range(max_bins, 1, -1):
        try:
            bins = pd.qcut(
                y_values,
                q=n_bins,
                labels=False,
                duplicates="drop",
            )
            bins = np.asarray(bins, dtype=int)
            counts = pd.Series(bins).value_counts()
            if counts.min() >= n_splits:
                return bins, n_bins
        except Exception:
            continue

    return np.zeros(len(y_values), dtype=int), 1


def build_patient_tables():
    valid_rows = []
    excluded_rows = []

    all_candidate_pids = sorted(
        set(X2_by_patient.keys())
        | set(Xstatic_by_patient.keys())
        | set(y_by_patient.keys()),
        key=str,
    )

    for pid in all_candidate_pids:
        reason = patient_invalid_reason(pid)
        if reason is not None:
            excluded_rows.append({
                "pid": pid,
                "exclusion_reason": reason,
            })
            continue

        yp = np.asarray(
            y_by_patient[pid],
            dtype=np.float32,
        )
        valid_rows.append({
            "pid": pid,
            "n_windows": int(len(yp)),
            "target_mean": float(np.mean(yp)),
            "target_median": float(np.median(yp)),
        })

    valid_df = pd.DataFrame(valid_rows)
    if len(valid_df):
        valid_df = valid_df.sort_values(
            "pid",
            key=lambda s: s.astype(str),
        ).reset_index(drop=True)

    excluded_df = pd.DataFrame(excluded_rows)
    if len(excluded_df):
        excluded_df = excluded_df.sort_values(
            "pid",
            key=lambda s: s.astype(str),
        ).reset_index(drop=True)

    return valid_df, excluded_df


patient_df, excluded_patient_df = build_patient_tables()
excluded_patient_df.to_csv(
    os.path.join(
        EXP_ROOT,
        "excluded_patients_with_reason.csv",
    ),
    index=False,
)

print("\nValid patients:", len(patient_df))
print("Excluded candidate patients:", len(excluded_patient_df))
print("Total windows:", int(patient_df["n_windows"].sum()))

if len(patient_df) < OUTER_SPLITS:
    raise ValueError("Not enough patients for outer CV")

outer_bins, used_bins = make_strat_bins(
    patient_df["target_mean"].values,
    OUTER_SPLITS,
    max_bins=10,
)
patient_df["outer_strat_bin"] = outer_bins
patient_df.to_csv(
    os.path.join(EXP_ROOT, "patient_summary_before_cv.csv"),
    index=False,
)

print("Outer stratification bins:", used_bins)
print(
    patient_df["outer_strat_bin"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 8. Optuna search space and experiment identity
# ============================================================
def sample_params(trial):
    architecture = trial.suggest_categorical(
        "architecture",
        OPTUNA_ARCHITECTURES,
    )
    head_type = ARCH_CONFIGS[architecture]["head_type"]

    params = {
        "architecture": architecture,
        "static_dim": trial.suggest_categorical(
            "static_dim",
            [8, 16, 32, 64],
        ),
        "dropout": trial.suggest_float(
            "dropout",
            0.0,
            0.50,
        ),
        "batch_size": trial.suggest_categorical(
            "batch_size",
            [32, 64, 128],
        ),
        "optimizer": trial.suggest_categorical(
            "optimizer",
            ["Adam", "AdamW"],
        ),
        "lr": trial.suggest_float(
            "lr",
            1e-4,
            5e-3,
            log=True,
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay",
            1e-7,
            3e-2,
            log=True,
        ),
        "smooth_l1_beta": trial.suggest_categorical(
            "smooth_l1_beta",
            [0.5, 1.0, 2.0],
        ),
        "grad_clip_norm": trial.suggest_categorical(
            "grad_clip_norm",
            [0.0, 1.0, 5.0],
        ),
        "patience": trial.suggest_categorical(
            "patience",
            [15, 20, 25, 30],
        ),
        "min_delta": trial.suggest_categorical(
            "min_delta",
            [1e-4, 5e-4, 1e-3],
        ),
        "scheduler_type": trial.suggest_categorical(
            "scheduler_type",
            ["none", "cosine"],
        ),
    }

    if head_type == "hidden":
        params["head_dim"] = trial.suggest_categorical(
            "head_dim",
            [16, 32, 64, 128],
        )
    else:
        params["head_dim"] = 0

    if params["scheduler_type"] == "cosine":
        params["cosine_eta_min_ratio"] = (
            trial.suggest_categorical(
                "cosine_eta_min_ratio",
                [0.001, 0.01, 0.05],
            )
        )
    else:
        params["cosine_eta_min_ratio"] = None

    return params


if "__file__" in globals() and os.path.isfile(__file__):
    with open(__file__, "rb") as f:
        SCRIPT_SHA256 = hashlib.sha256(f.read()).hexdigest()
else:
    SCRIPT_SHA256 = "unavailable"

data_stat = os.stat(DATA_PKL)
dataset_fingerprint = {
    "absolute_path": os.path.abspath(DATA_PKL),
    "size_bytes": int(data_stat.st_size),
    "mtime_ns": int(data_stat.st_mtime_ns),
}

run_config = {
    "code_version": CODE_VERSION,
    "script_sha256": SCRIPT_SHA256,
    "dataset_label": DATASET_LABEL,
    "dataset_fingerprint": dataset_fingerprint,
    "exp_root": os.path.abspath(EXP_ROOT),
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "n_trials_per_outer": N_TRIALS_PER_OUTER,
    "max_epochs": MAX_EPOCHS,
    "seed": SEED,
    "final_refit_seeds": FINAL_REFIT_SEEDS,
    "final_refit_training": (
        "all_outer_train_fixed_epoch_"
        "patient_balanced_multiseed_ensemble"
    ),
    "dynamic_normalization": NORMALIZATION_MODE,
    "patient_balanced_training": PATIENT_BALANCED_TRAINING,
    "training_dataset_lifecycle": (
        "one_WindowDataset_per_model_fit_"
        "epoch_recreates_sampler_and_loader_only"
    ),
    "patient_balanced_normalization": (
        PATIENT_BALANCED_NORMALIZATION
    ),
    "scheduler": (
        "none_or_cosine_epoch_based_"
        "Tmax_equals_MAX_EPOCHS"
    ),
    "mask_normalization": "unchanged",
    "outlier_top_fraction": OUTLIER_TOP_FRACTION,
    "outlier_selection": (
        "strict_nlargest_ceil_N_times_fraction"
    ),
    "static_features": STATIC_FEATURES,
    "dynamic_features": dynamic_feature_cols,
    "feature_selection_policy": (
        "strict_exact_ordered_14_dynamic_plus_3_static_usa_common_features"
    ),
    "source_dataset_fingerprint": source_fingerprint,
    "common_dataset_metadata_hash": common_metadata_hash,
    "common_dataset_path": os.path.abspath(COMMON_DATA_PKL),
    "usa_features_reported": USA_FEATURES_REPORTED,
    "target_only_features_not_in_input": TARGET_ONLY_FEATURES,
    "excluded_usa_features": EXCLUDED_USA_FEATURES,
    "forbidden_features_exact": FORBIDDEN_FEATURES,
    "strict_no_forbidden_features": STRICT_NO_FORBIDDEN_FEATURES,
    "outer_fold_source": os.path.abspath(MAIN_OUTER_ASSIGNMENT_CSV),
    "architecture_candidates": {
        name: ARCH_CONFIGS[name]
        for name in OPTUNA_ARCHITECTURES
    },
    "static_only_baseline_excluded_from_primary_search": True,
}

EXPERIMENT_HASH = stable_hash(run_config)
run_config["experiment_hash"] = EXPERIMENT_HASH

signature_path = os.path.join(
    EXP_ROOT,
    "experiment_signature.json",
)
if os.path.isfile(signature_path):
    with open(signature_path, "r", encoding="utf-8") as f:
        previous_signature = json.load(f)
    if (
        previous_signature.get("experiment_hash")
        != EXPERIMENT_HASH
    ):
        raise RuntimeError(
            "EXP_ROOT contains an incompatible experiment. "
            "Use a new EXP_ROOT or restore the matching configuration. "
            f"Existing hash={previous_signature.get('experiment_hash')}, "
            f"current hash={EXPERIMENT_HASH}"
        )
else:
    atomic_json_dump(run_config, signature_path)

atomic_json_dump(
    run_config,
    os.path.join(EXP_ROOT, "run_config.json"),
)

print("Experiment hash:", EXPERIMENT_HASH)
print(
    "Feature policy: strict ordered Sweden-USA common set: "
    "14 dynamic signal features + GA/sex/birthweight static features."
)
print(
    "Leakage control: feats__pna_days is never a model input; on USA it is "
    "used only to construct true PMA = GA + PNA/7."
)
print(
    "Audit warning: exact names and numerical scales are checked automatically, "
    "but BTB preprocessing and entropy definitions still require metadata review."
)
print(
    "Normalization:",
    NORMALIZATION_MODE,
)
print(
    "Patient-balanced training:",
    PATIENT_BALANCED_TRAINING,
)

# ============================================================
# 9. Aggregation utilities for final outer predictions
# ============================================================
def aggregate_outer_window_predictions(df):
    if len(df) == 0:
        return pd.DataFrame()
    group_cols = ["pid", "window_idx", "window_id", "outer_fold"]
    agg = (
        df.groupby(group_cols, as_index=False)
        .agg(
            true_window=("true_window", "mean"),
            pred_window_mean=("pred_window", "mean"),
            pred_window_std=("pred_window", "std"),
            n_model_repeats=("pred_window", "size"),
            architecture=("architecture", lambda x: ";".join(sorted(set(map(str, x))))),
        )
    )
    agg["pred_window_std"] = agg["pred_window_std"].fillna(0.0)
    agg["error_window"] = agg["pred_window_mean"] - agg["true_window"]
    return agg


def aggregate_patient_from_windows(window_df):
    if len(window_df) == 0:
        return pd.DataFrame(), {"patient_mae": np.nan, "patient_rmse": np.nan, "n_patients": 0}
    pat = (
        window_df.groupby("pid", as_index=False)
        .agg(
            true_patient=("true_window", "mean"),
            pred_patient_mean=("pred_window_mean", "mean"),
            pred_patient_std_across_windows=("pred_window_mean", "std"),
            n_windows=("window_idx", "size"),
            outer_fold=("outer_fold", "first"),
            n_model_repeats=("n_model_repeats", "first"),
            architecture=("architecture", "first"),
        )
    )
    pat["pred_patient_std_across_windows"] = pat["pred_patient_std_across_windows"].fillna(0.0)
    pat["error_patient"] = pat["pred_patient_mean"] - pat["true_patient"]
    pat["abs_error"] = np.abs(pat["error_patient"])
    pat["sq_error"] = pat["error_patient"] ** 2
    summary = {
        "patient_mae": float(pat["abs_error"].mean()),
        "patient_rmse": float(np.sqrt(pat["sq_error"].mean())),
        "n_patients": int(len(pat)),
    }
    return pat, summary

# ============================================================
# 10. Run outer nested CV with end-to-end resume
# ============================================================
all_outer_window_preds = []
all_outer_patient_preds_individual_models = []
outer_summary_rows = []
fold_assignment_rows = []

if not os.path.isfile(MAIN_OUTER_ASSIGNMENT_CSV):
    raise FileNotFoundError(
        "The completed full-feature CNN outer-fold assignment is required: "
        f"{MAIN_OUTER_ASSIGNMENT_CSV}"
    )

main_assignment = pd.read_csv(MAIN_OUTER_ASSIGNMENT_CSV, dtype={"pid": str})
required_assignment_cols = {"pid", "outer_fold", "role"}
if not required_assignment_cols.issubset(main_assignment.columns):
    raise ValueError(
        "MAIN_OUTER_ASSIGNMENT_CSV is missing columns: "
        f"{sorted(required_assignment_cols - set(main_assignment.columns))}"
    )
main_assignment["pid_key"] = main_assignment["pid"].map(normalize_pid_text)
main_assignment["outer_fold"] = pd.to_numeric(
    main_assignment["outer_fold"], errors="raise"
).astype(int)

patient_pid_keys = patient_df["pid"].map(normalize_pid_text)
if patient_pid_keys.duplicated().any():
    duplicated = patient_pid_keys[patient_pid_keys.duplicated(keep=False)].tolist()
    raise RuntimeError(f"Normalized patient IDs are not unique: {duplicated[:20]}")
pid_lookup = dict(zip(patient_pid_keys, patient_df["pid"]))
all_pid_keys = set(pid_lookup)

main_test = main_assignment[main_assignment["role"] == "outer_test"].copy()
main_test = main_test[main_test["pid_key"].isin(all_pid_keys)]
assignment_counts = main_test.groupby("pid_key").size()
if set(assignment_counts.index) != all_pid_keys or not (assignment_counts == 1).all():
    missing = sorted(all_pid_keys - set(assignment_counts.index))[:20]
    duplicate = assignment_counts[assignment_counts != 1].to_dict()
    raise RuntimeError(
        "Full-CNN outer-test assignment does not cover the common-feature cohort exactly. "
        f"Missing={missing}, non-single counts={dict(list(duplicate.items())[:20])}"
    )

outer_split_plan = {}
for outer_fold in range(1, OUTER_SPLITS + 1):
    test_keys = set(
        main_test.loc[main_test["outer_fold"] == outer_fold, "pid_key"].tolist()
    )
    if not test_keys:
        raise RuntimeError(f"No outer-test patients found for fold {outer_fold}")
    train_keys = all_pid_keys - test_keys
    outer_split_plan[outer_fold] = {
        "train_pids": [pid_lookup[k] for k in sorted(train_keys)],
        "test_pids": [pid_lookup[k] for k in sorted(test_keys)],
    }

pd.DataFrame(
    [
        {"pid": pid_lookup[key], "outer_fold": int(row.outer_fold), "role": "outer_test"}
        for row in main_test.itertuples(index=False)
        for key in [row.pid_key]
    ]
).to_csv(
    os.path.join(EXP_ROOT, "reused_main_CNN_outer_test_assignment.csv"),
    index=False,
)


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def marker_matches_experiment(marker_path):
    if not os.path.isfile(marker_path):
        return False
    marker = read_json(marker_path)
    if marker.get("experiment_hash") != EXPERIMENT_HASH:
        raise RuntimeError(
            f"Incompatible completion marker: {marker_path}"
        )
    return True


for outer_fold in range(1, OUTER_SPLITS + 1):
    outer_dir = os.path.join(
        EXP_ROOT,
        f"outer_fold_{outer_fold:02d}",
    )
    os.makedirs(outer_dir, exist_ok=True)

    outer_train_pids = outer_split_plan[outer_fold]["train_pids"]
    outer_test_pids = outer_split_plan[outer_fold]["test_pids"]
    outer_train_keys = {normalize_pid_text(x) for x in outer_train_pids}
    outer_test_keys = {normalize_pid_text(x) for x in outer_test_pids}
    outer_train_df = patient_df[
        patient_df["pid"].map(normalize_pid_text).isin(outer_train_keys)
    ].copy().reset_index(drop=True)
    outer_test_df = patient_df[
        patient_df["pid"].map(normalize_pid_text).isin(outer_test_keys)
    ].copy().reset_index(drop=True)

    if set(outer_train_keys) & set(outer_test_keys):
        raise RuntimeError(f"Outer fold {outer_fold}: train/test patient overlap")
    if len(outer_train_df) + len(outer_test_df) != len(patient_df):
        raise RuntimeError(f"Outer fold {outer_fold}: incomplete patient coverage")

    for pid in outer_train_pids:
        fold_assignment_rows.append({
            "pid": pid,
            "outer_fold": outer_fold,
            "role": "outer_train",
        })
    for pid in outer_test_pids:
        fold_assignment_rows.append({
            "pid": pid,
            "outer_fold": outer_fold,
            "role": "outer_test",
        })

    outer_fold_identity = {
        "experiment_hash": EXPERIMENT_HASH,
        "outer_fold": int(outer_fold),
        "outer_train_pids_hash": stable_hash(
            [str(x) for x in outer_train_pids]
        ),
        "outer_test_pids_hash": stable_hash(
            [str(x) for x in outer_test_pids]
        ),
    }
    outer_fold_hash = stable_hash(outer_fold_identity)

    print("\n" + "=" * 120)
    print(f"OUTER FOLD {outer_fold}/{OUTER_SPLITS}")
    print(
        f"outer_train patients={len(outer_train_pids)}, "
        f"outer_test patients={len(outer_test_pids)}"
    )
    print("=" * 120)

    outer_complete_marker = os.path.join(
        outer_dir,
        "OUTER_FOLD_COMPLETE.json",
    )
    outer_window_all_csv = os.path.join(
        outer_dir,
        "outer_test_window_predictions_all_full_refit_seeds.csv",
    )
    outer_patient_all_csv = os.path.join(
        outer_dir,
        "outer_test_patient_predictions_all_full_refit_seeds.csv",
    )

    if marker_matches_experiment(outer_complete_marker):
        marker = read_json(outer_complete_marker)
        if marker.get("outer_fold_hash") != outer_fold_hash:
            raise RuntimeError(
                f"Outer fold {outer_fold} split identity mismatch."
            )
        required = [
            outer_window_all_csv,
            outer_patient_all_csv,
        ]
        missing = [p for p in required if not os.path.isfile(p)]
        if missing:
            raise RuntimeError(
                f"Outer fold {outer_fold} marker exists but files "
                f"are missing: {missing}"
            )

        loaded_w = pd.read_csv(outer_window_all_csv)
        loaded_p = pd.read_csv(outer_patient_all_csv)
        all_outer_window_preds.append(loaded_w)
        all_outer_patient_preds_individual_models.append(
            loaded_p
        )
        outer_summary_rows.append(
            marker["outer_summary_row"]
        )

        print(
            f"Outer fold {outer_fold} already complete; "
            "loaded saved predictions and skipped Optuna/refit."
        )
        continue

    # Inner folds are created only from outer_train patients.
    inner_bins, inner_used_bins = make_strat_bins(
        outer_train_df["target_mean"].values,
        n_splits=INNER_SPLITS,
        max_bins=8,
    )
    outer_train_df["inner_strat_bin"] = inner_bins
    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED + outer_fold,
    )
    inner_splits = []

    for inner_fold, (
        inner_tr_idx,
        inner_val_idx,
    ) in enumerate(
        inner_cv.split(
            outer_train_df["pid"].values,
            outer_train_df["inner_strat_bin"].values,
        ),
        start=1,
    ):
        inner_train_pids = (
            outer_train_df.iloc[inner_tr_idx]["pid"]
            .tolist()
        )
        inner_val_pids = (
            outer_train_df.iloc[inner_val_idx]["pid"]
            .tolist()
        )
        inner_splits.append(
            (
                inner_fold,
                inner_train_pids,
                inner_val_pids,
            )
        )

    pd.DataFrame({
        "pid": outer_train_df["pid"],
        "target_mean": outer_train_df["target_mean"],
        "inner_strat_bin": (
            outer_train_df["inner_strat_bin"]
        ),
    }).to_csv(
        os.path.join(
            outer_dir,
            "outer_train_inner_bins.csv",
        ),
        index=False,
    )

    def objective(trial):
        params = sample_params(trial)
        val_maes = []
        val_rmses = []
        n_params_seen = None
        best_epochs = []

        try:
            for (
                inner_fold,
                inner_train_pids,
                inner_val_pids,
            ) in inner_splits:
                # The same outer/inner seed is used across all trials.
                run_seed = (
                    SEED
                    + outer_fold * 100
                    + inner_fold
                )
                ckpt_path = None
                if SAVE_TRIAL_CHECKPOINTS:
                    ckpt_path = os.path.join(
                        outer_dir,
                        f"trial_{trial.number:03d}_"
                        f"inner{inner_fold}_model.pt",
                    )

                res = train_one_model(
                    run_name=(
                        f"outer{outer_fold}_"
                        f"trial{trial.number:03d}_"
                        f"inner{inner_fold}"
                    ),
                    train_pids=inner_train_pids,
                    val_pids=inner_val_pids,
                    test_pids=None,
                    params=params,
                    max_epochs=MAX_EPOCHS,
                    save_checkpoint_path=ckpt_path,
                    evaluate_test=False,
                    verbose=False,
                    run_seed=run_seed,
                    meta={
                        "outer_fold": outer_fold,
                        "inner_fold": inner_fold,
                        "trial": trial.number,
                        "architecture": params["architecture"],
                    },
                )

                val_maes.append(res["val_patient_mae"])
                val_rmses.append(res["val_patient_rmse"])
                best_epochs.append(res["best_epoch"])
                n_params_seen = res["n_params"]

                interim = float(np.mean(val_maes))
                trial.report(interim, step=inner_fold)
                cleanup()

                if trial.should_prune():
                    raise optuna.TrialPruned()

        except RuntimeError:
            cleanup()
            raise

        trial.set_user_attr(
            "inner_val_patient_MAE_mean",
            float(np.mean(val_maes)),
        )
        trial.set_user_attr(
            "inner_val_patient_MAE_std",
            float(np.std(val_maes, ddof=0)),
        )
        trial.set_user_attr(
            "inner_val_patient_RMSE_mean",
            float(np.mean(val_rmses)),
        )
        trial.set_user_attr(
            "n_params",
            int(n_params_seen),
        )
        trial.set_user_attr(
            "best_epoch_mean",
            float(np.mean(best_epochs)),
        )
        trial.set_user_attr(
            "best_epoch_median",
            float(np.median(best_epochs)),
        )
        trial.set_user_attr(
            "best_epochs_by_inner_fold",
            [int(x) for x in best_epochs],
        )
        trial.set_user_attr(
            "architecture",
            params["architecture"],
        )
        trial.set_user_attr(
            "channels",
            str(
                ARCH_CONFIGS[
                    params["architecture"]
                ]["channels"]
            ),
        )
        trial.set_user_attr(
            "head_type",
            ARCH_CONFIGS[
                params["architecture"]
            ]["head_type"],
        )

        return float(np.mean(val_maes))

    sampler = optuna.samplers.TPESampler(
        seed=SEED + outer_fold,
    )
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=8,
        n_warmup_steps=2,
    )
    study = optuna.create_study(
        study_name=(
            f"nested_arch_optuna_outer_"
            f"{outer_fold:02d}"
        ),
        storage=(
            "sqlite:///"
            + os.path.join(
                outer_dir,
                "optuna_study.db",
            )
        ),
        load_if_exists=True,
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
    )

    existing_experiment_hash = study.user_attrs.get(
        "experiment_hash"
    )
    existing_outer_fold_hash = study.user_attrs.get(
        "outer_fold_hash"
    )
    if existing_experiment_hash is None:
        if len(study.trials) > 0:
            raise RuntimeError(
                f"Outer fold {outer_fold} study contains trials but "
                "has no v4 experiment signature. Use a new EXP_ROOT."
            )
        study.set_user_attr(
            "experiment_hash",
            EXPERIMENT_HASH,
        )
        study.set_user_attr(
            "outer_fold_hash",
            outer_fold_hash,
        )
    elif (
        existing_experiment_hash != EXPERIMENT_HASH
        or existing_outer_fold_hash != outer_fold_hash
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold} Optuna database is "
            "incompatible with this experiment/split."
        )

    def get_trial_state_counts(current_study):
        state_counts = {
            "COMPLETE": 0,
            "PRUNED": 0,
            "FAIL": 0,
            "RUNNING": 0,
            "WAITING": 0,
        }
        for frozen_trial in current_study.trials:
            state_name = frozen_trial.state.name
            state_counts[state_name] = (
                state_counts.get(state_name, 0) + 1
            )
        return state_counts

    topup_round = 0
    while True:
        state_counts = get_trial_state_counts(study)
        n_counted_trials = (
            state_counts.get("COMPLETE", 0)
            + state_counts.get("PRUNED", 0)
        )
        if n_counted_trials > N_TRIALS_PER_OUTER:
            raise RuntimeError(
                f"Outer fold {outer_fold} already contains "
                f"{n_counted_trials} COMPLETE+PRUNED trials, exceeding "
                f"the configured target {N_TRIALS_PER_OUTER}. "
                "Use a new EXP_ROOT for an exactly controlled run."
            )

        remaining_trials = max(
            0,
            N_TRIALS_PER_OUTER - n_counted_trials,
        )

        print(
            f"Optuna outer {outer_fold} resume status | "
            f"COMPLETE={state_counts.get('COMPLETE', 0)}, "
            f"PRUNED={state_counts.get('PRUNED', 0)}, "
            f"FAIL={state_counts.get('FAIL', 0)}, "
            f"RUNNING={state_counts.get('RUNNING', 0)}, "
            f"WAITING={state_counts.get('WAITING', 0)}, "
            f"COUNTED={n_counted_trials}/"
            f"{N_TRIALS_PER_OUTER}, "
            f"REMAINING={remaining_trials}"
        )

        if remaining_trials <= 0:
            break

        counted_before = n_counted_trials
        topup_round += 1

        study.optimize(
            objective,
            n_trials=remaining_trials,
            gc_after_trial=True,
            # Runtime errors such as a single CUDA OOM become FAIL
            # trials; they do not count toward the target and are replaced.
            catch=(RuntimeError,),
        )

        state_counts_after = get_trial_state_counts(
            study
        )
        counted_after = (
            state_counts_after.get("COMPLETE", 0)
            + state_counts_after.get("PRUNED", 0)
        )
        if counted_after <= counted_before:
            raise RuntimeError(
                f"Optuna outer {outer_fold}: no progress "
                "toward the COMPLETE+PRUNED target. "
                "Inspect failed trials/logs."
            )

    trial_rows = []
    for t in study.trials:
        row = {
            "trial": t.number,
            "state": str(t.state),
            "objective_inner_mean_val_patient_MAE": (
                t.value
                if t.value is not None
                else np.nan
            ),
        }
        row.update(t.params)
        row.update(t.user_attrs)
        trial_rows.append(row)

    trial_df = pd.DataFrame(trial_rows).sort_values(
        "objective_inner_mean_val_patient_MAE",
        ascending=True,
        na_position="last",
    )
    trial_df.to_csv(
        os.path.join(outer_dir, "optuna_trials.csv"),
        index=False,
    )

    best_params = dict(study.best_trial.params)
    if (
        ARCH_CONFIGS[
            best_params["architecture"]
        ]["head_type"]
        != "hidden"
    ):
        best_params["head_dim"] = 0
    else:
        best_params["head_dim"] = best_params.get(
            "head_dim",
            32,
        )

    if best_params.get("scheduler_type", "none") == "none":
        best_params["cosine_eta_min_ratio"] = None
    else:
        best_params["cosine_eta_min_ratio"] = (
            best_params.get(
                "cosine_eta_min_ratio",
                0.01,
            )
        )

    atomic_json_dump(
        {
            "experiment_hash": EXPERIMENT_HASH,
            "outer_fold_hash": outer_fold_hash,
            "best_trial_number": study.best_trial.number,
            "best_value_inner_mean_val_patient_MAE": (
                study.best_value
            ),
            "best_params": best_params,
            "best_trial_user_attrs": (
                study.best_trial.user_attrs
            ),
        },
        os.path.join(outer_dir, "best_params.json"),
    )

    best_epoch_median = study.best_trial.user_attrs.get(
        "best_epoch_median",
        study.best_trial.user_attrs.get(
            "best_epoch_mean",
            1.0,
        ),
    )
    final_refit_epochs = int(
        np.clip(
            int(round(float(best_epoch_median))),
            1,
            MAX_EPOCHS,
        )
    )

    print(
        f"Best outer {outer_fold}: "
        f"trial={study.best_trial.number}, "
        f"inner MAE={study.best_value:.4f}"
    )
    print("Best params:", best_params)
    print(
        f"Final refit epochs: {final_refit_epochs}"
    )

    outer_test_window_dfs = []
    outer_test_patient_dfs = []
    final_model_rows = []

    for model_repeat, base_seed in enumerate(
        FINAL_REFIT_SEEDS,
        start=1,
    ):
        run_seed = int(
            base_seed + outer_fold * 100000
        )
        seed_tag = f"seed{base_seed}"
        seed_marker_path = os.path.join(
            outer_dir,
            f"{seed_tag}_COMPLETE.json",
        )
        seed_window_csv = os.path.join(
            outer_dir,
            f"{seed_tag}_outer_test_window_predictions.csv",
        )
        seed_patient_csv = os.path.join(
            outer_dir,
            f"{seed_tag}_outer_test_patient_predictions.csv",
        )
        seed_history_csv = os.path.join(
            outer_dir,
            f"full_outer_train_{seed_tag}_history.csv",
        )
        final_ckpt_path = (
            os.path.join(
                outer_dir,
                f"full_outer_train_{seed_tag}_model.pt",
            )
            if SAVE_BEST_CHECKPOINTS
            else None
        )
        progress_ckpt_path = os.path.join(
            outer_dir,
            f"full_outer_train_{seed_tag}_PROGRESS.pt",
        )

        if marker_matches_experiment(seed_marker_path):
            marker = read_json(seed_marker_path)
            if marker.get("outer_fold_hash") != outer_fold_hash:
                raise RuntimeError(
                    f"{seed_tag}: outer fold identity mismatch."
                )
            required = [
                seed_window_csv,
                seed_patient_csv,
                seed_history_csv,
            ]
            missing = [
                p for p in required
                if not os.path.isfile(p)
            ]
            if missing:
                raise RuntimeError(
                    f"{seed_tag} marker exists but files "
                    f"are missing: {missing}"
                )

            test_w = pd.read_csv(seed_window_csv)
            test_p = pd.read_csv(seed_patient_csv)
            model_summary_row = marker[
                "model_summary_row"
            ]

            print(
                f"Outer {outer_fold} {seed_tag} already "
                "complete; loaded saved predictions."
            )
        else:
            res = train_full_outer_model(
                run_name=(
                    f"outer{outer_fold}_FULL_{seed_tag}"
                ),
                train_pids=outer_train_pids,
                test_pids=outer_test_pids,
                params=best_params,
                n_epochs=final_refit_epochs,
                save_checkpoint_path=final_ckpt_path,
                resume_checkpoint_path=progress_ckpt_path,
                verbose=True,
                run_seed=run_seed,
                meta={
                    "outer_fold": outer_fold,
                    "model_repeat": model_repeat,
                    "base_seed": int(base_seed),
                    "run_seed": run_seed,
                    "best_trial": study.best_trial.number,
                    "architecture": (
                        best_params["architecture"]
                    ),
                    "training_mode": (
                        "full_outer_train_fixed_epoch_"
                        "patient_balanced"
                    ),
                },
                experiment_hash=EXPERIMENT_HASH,
            )

            test_w = res["test_window_df"].copy()
            test_p = res["test_patient_df"].copy()
            test_w.to_csv(
                seed_window_csv,
                index=False,
            )
            test_p.to_csv(
                seed_patient_csv,
                index=False,
            )
            res["history"].to_csv(
                seed_history_csv,
                index=False,
            )

            model_summary_row = {
                "outer_fold": outer_fold,
                "model_repeat": model_repeat,
                "base_seed": int(base_seed),
                "run_seed": run_seed,
                "best_trial": study.best_trial.number,
                "architecture": (
                    best_params["architecture"]
                ),
                "n_params": res["n_params"],
                "final_refit_epochs": res["n_epochs"],
                "n_outer_train_patients": len(
                    outer_train_pids
                ),
                "n_outer_test_patients": len(
                    outer_test_pids
                ),
                "outer_test_patient_mae_single_seed": (
                    res["test_patient_mae"]
                ),
                "outer_test_patient_rmse_single_seed": (
                    res["test_patient_rmse"]
                ),
                "outer_test_window_mae_single_seed": (
                    res["test_window_mae"]
                ),
                "outer_test_window_rmse_single_seed": (
                    res["test_window_rmse"]
                ),
                "checkpoint": final_ckpt_path,
            }

            atomic_json_dump(
                {
                    "experiment_hash": EXPERIMENT_HASH,
                    "outer_fold_hash": outer_fold_hash,
                    "model_summary_row": model_summary_row,
                },
                seed_marker_path,
            )

        outer_test_window_dfs.append(test_w)
        outer_test_patient_dfs.append(test_p)
        all_outer_window_preds.append(test_w)
        all_outer_patient_preds_individual_models.append(
            test_p
        )
        final_model_rows.append(model_summary_row)
        cleanup()

    final_model_df = pd.DataFrame(final_model_rows)
    final_model_df.to_csv(
        os.path.join(
            outer_dir,
            "final_full_outer_refit_models_summary.csv",
        ),
        index=False,
    )

    outer_test_window_all = pd.concat(
        outer_test_window_dfs,
        ignore_index=True,
    )
    outer_test_window_all.to_csv(
        outer_window_all_csv,
        index=False,
    )

    outer_test_patient_all = pd.concat(
        outer_test_patient_dfs,
        ignore_index=True,
    )
    outer_test_patient_all.to_csv(
        outer_patient_all_csv,
        index=False,
    )

    outer_test_window_agg = (
        aggregate_outer_window_predictions(
            outer_test_window_all
        )
    )
    outer_test_window_agg.to_csv(
        os.path.join(
            outer_dir,
            "outer_test_window_predictions_3seed_ensemble.csv",
        ),
        index=False,
    )

    (
        outer_test_patient_agg,
        outer_summary,
    ) = aggregate_patient_from_windows(
        outer_test_window_agg
    )
    outer_test_patient_agg.to_csv(
        os.path.join(
            outer_dir,
            "outer_test_patient_predictions_3seed_ensemble.csv",
        ),
        index=False,
    )

    outer_summary_row = {
        "outer_fold": outer_fold,
        "n_outer_train_patients": len(
            outer_train_pids
        ),
        "n_outer_test_patients": len(
            outer_test_pids
        ),
        "best_trial": study.best_trial.number,
        "best_inner_mean_val_patient_mae": float(
            study.best_value
        ),
        "best_inner_epoch_mean": float(
            study.best_trial.user_attrs.get(
                "best_epoch_mean",
                np.nan,
            )
        ),
        "best_inner_epoch_median": float(
            best_epoch_median
        ),
        "final_refit_epochs": int(
            final_refit_epochs
        ),
        "n_final_refit_seeds": int(
            len(FINAL_REFIT_SEEDS)
        ),
        "final_refit_seeds": ";".join(
            map(str, FINAL_REFIT_SEEDS)
        ),
        "selected_architecture": (
            best_params["architecture"]
        ),
        "selected_static_dim": (
            best_params["static_dim"]
        ),
        "selected_dropout": (
            best_params["dropout"]
        ),
        "selected_head_dim": best_params.get(
            "head_dim",
            0,
        ),
        "selected_batch_size": (
            best_params["batch_size"]
        ),
        "selected_optimizer": (
            best_params["optimizer"]
        ),
        "selected_lr": best_params["lr"],
        "selected_weight_decay": (
            best_params["weight_decay"]
        ),
        "selected_scheduler_type": (
            best_params["scheduler_type"]
        ),
        "single_seed_patient_mae_mean": float(
            final_model_df[
                "outer_test_patient_mae_single_seed"
            ].mean()
        ),
        "single_seed_patient_mae_std": float(
            final_model_df[
                "outer_test_patient_mae_single_seed"
            ].std(ddof=0)
        ),
        "outer_test_patient_mae_ensemble": (
            outer_summary["patient_mae"]
        ),
        "outer_test_patient_rmse_ensemble": (
            outer_summary["patient_rmse"]
        ),
        "outer_test_n_patients": (
            outer_summary["n_patients"]
        ),
    }
    outer_summary_rows.append(outer_summary_row)

    pd.DataFrame(outer_summary_rows).to_csv(
        os.path.join(
            EXP_ROOT,
            "outer_fold_summary_running.csv",
        ),
        index=False,
    )

    atomic_json_dump(
        {
            "experiment_hash": EXPERIMENT_HASH,
            "outer_fold_hash": outer_fold_hash,
            "outer_summary_row": outer_summary_row,
        },
        outer_complete_marker,
    )


# ============================================================
# 11. Final pooled nested OOF results and strict integrity checks
# ============================================================
fold_assignment_df = pd.DataFrame(
    fold_assignment_rows
)
fold_assignment_df.to_csv(
    os.path.join(
        EXP_ROOT,
        "outer_fold_assignment_long.csv",
    ),
    index=False,
)

outer_test_assignment_counts = (
    fold_assignment_df[
        fold_assignment_df["role"] == "outer_test"
    ]
    .groupby("pid")
    .size()
)
if (
    len(outer_test_assignment_counts) != len(patient_df)
    or not (
        outer_test_assignment_counts == 1
    ).all()
):
    raise RuntimeError(
        "Each valid patient must appear in outer-test exactly once."
    )

outer_summary_df = pd.DataFrame(
    outer_summary_rows
).sort_values("outer_fold")
outer_summary_df.to_csv(
    os.path.join(
        EXP_ROOT,
        "outer_fold_summary.csv",
    ),
    index=False,
)

all_window_individual = pd.concat(
    all_outer_window_preds,
    ignore_index=True,
)
all_window_individual.to_csv(
    os.path.join(
        EXP_ROOT,
        "ALL_outer_test_window_predictions_"
        "individual_full_refit_seeds.csv",
    ),
    index=False,
)

individual_key = [
    "pid",
    "window_idx",
    "outer_fold",
]
seed_count_series = (
    all_window_individual
    .groupby(individual_key)
    .size()
)
bad_seed_counts = seed_count_series[
    seed_count_series != len(FINAL_REFIT_SEEDS)
]
if len(bad_seed_counts):
    raise RuntimeError(
        "Some outer-test windows do not have exactly "
        f"{len(FINAL_REFIT_SEEDS)} seed predictions."
    )

window_fold_counts = (
    all_window_individual
    .groupby(["pid", "window_idx"])["outer_fold"]
    .nunique()
)
if not (window_fold_counts == 1).all():
    raise RuntimeError(
        "A window appears in more than one outer fold."
    )

true_value_counts = (
    all_window_individual
    .groupby(individual_key)["true_window"]
    .nunique()
)
if not (true_value_counts == 1).all():
    raise RuntimeError(
        "Inconsistent true_window values across seed predictions."
    )

final_window_df = aggregate_outer_window_predictions(
    all_window_individual
)
final_window_df.to_csv(
    os.path.join(
        EXP_ROOT,
        "FINAL_nested_oof_window_predictions.csv",
    ),
    index=False,
)

final_patient_df, final_summary = (
    aggregate_patient_from_windows(
        final_window_df
    )
)
final_patient_df = (
    final_patient_df
    .sort_values("abs_error", ascending=False)
    .reset_index(drop=True)
)
final_patient_df.to_csv(
    os.path.join(
        EXP_ROOT,
        "FINAL_nested_oof_patient_predictions.csv",
    ),
    index=False,
)

expected_pids = {
    str(x) for x in patient_df["pid"].tolist()
}
observed_pids = {
    str(x) for x in final_patient_df["pid"].tolist()
}
missing_oof_pids = sorted(
    expected_pids - observed_pids
)
unexpected_oof_pids = sorted(
    observed_pids - expected_pids
)

patient_outer_fold_counts = (
    all_window_individual
    .groupby("pid")["outer_fold"]
    .nunique()
)
exactly_one_fold_per_patient = bool(
    len(patient_outer_fold_counts) == len(patient_df)
    and (patient_outer_fold_counts == 1).all()
)

oof_integrity = {
    "experiment_hash": EXPERIMENT_HASH,
    "expected_valid_patients": int(
        len(expected_pids)
    ),
    "observed_oof_patients": int(
        len(observed_pids)
    ),
    "all_patients_covered": bool(
        not missing_oof_pids
        and not unexpected_oof_pids
    ),
    "each_patient_outer_test_exactly_once": bool(
        (outer_test_assignment_counts == 1).all()
    ),
    "each_patient_in_one_prediction_fold": (
        exactly_one_fold_per_patient
    ),
    "each_window_has_expected_seed_predictions": bool(
        len(bad_seed_counts) == 0
    ),
    "missing_pids": missing_oof_pids,
    "unexpected_pids": unexpected_oof_pids,
}
atomic_json_dump(
    oof_integrity,
    os.path.join(
        EXP_ROOT,
        "FINAL_oof_integrity_check.json",
    ),
)

if not all([
    oof_integrity["all_patients_covered"],
    oof_integrity[
        "each_patient_outer_test_exactly_once"
    ],
    oof_integrity[
        "each_patient_in_one_prediction_fold"
    ],
    oof_integrity[
        "each_window_has_expected_seed_predictions"
    ],
]):
    raise RuntimeError(
        "Nested OOF integrity checks failed. "
        "See FINAL_oof_integrity_check.json."
    )

# Strict top 10%: exactly ceil(N * 0.10) patients.
n_outliers = max(
    1,
    int(
        math.ceil(
            len(final_patient_df)
            * OUTLIER_TOP_FRACTION
        )
    ),
)
outlier_patients = (
    final_patient_df
    .nlargest(n_outliers, "abs_error")
    .reset_index(drop=True)
)
outlier_threshold = float(
    outlier_patients["abs_error"].min()
)
outlier_patients["outlier_rank"] = np.arange(
    1,
    len(outlier_patients) + 1,
)
outlier_patients[
    "outlier_threshold_abs_error"
] = outlier_threshold
outlier_patients["outlier_definition"] = (
    f"strict_highest_{OUTLIER_TOP_FRACTION:.0%}_"
    "patient_absolute_oof_error"
)
outlier_patients.to_csv(
    os.path.join(
        EXP_ROOT,
        "FINAL_top10percent_outlier_patients.csv",
    ),
    index=False,
)

arch_freq = (
    outer_summary_df["selected_architecture"]
    .value_counts()
    .reset_index()
)
arch_freq.columns = [
    "selected_architecture",
    "n_outer_folds",
]
arch_freq.to_csv(
    os.path.join(
        EXP_ROOT,
        "selected_architecture_frequency.csv",
    ),
    index=False,
)

final_summary.update({
    "experiment_hash": EXPERIMENT_HASH,
    "dataset_label": DATASET_LABEL,
    "data_pkl": DATA_PKL,
    "n_outer_splits": OUTER_SPLITS,
    "n_inner_splits": INNER_SPLITS,
    "n_trials_per_outer_counting_complete_plus_pruned": (
        N_TRIALS_PER_OUTER
    ),
    "final_refit_seeds": FINAL_REFIT_SEEDS,
    "final_refit_training": (
        "all_outer_train_fixed_epoch_"
        "patient_balanced_multiseed_ensemble"
    ),
    "dynamic_normalization": NORMALIZATION_MODE,
    "patient_balanced_training": (
        PATIENT_BALANCED_TRAINING
    ),
    "patient_balanced_normalization": (
        PATIENT_BALANCED_NORMALIZATION
    ),
    "scheduler": (
        "none_or_cosine_epoch_based_"
        "same_trajectory_inner_and_final"
    ),
    "outlier_top_fraction": (
        OUTLIER_TOP_FRACTION
    ),
    "outlier_selection": (
        "strict_nlargest_ceil_N_times_fraction"
    ),
    "outlier_abs_error_threshold": (
        outlier_threshold
    ),
    "n_outlier_patients": int(
        len(outlier_patients)
    ),
    "total_patients": int(
        final_patient_df["pid"].nunique()
    ),
    "total_windows": int(
        len(final_window_df)
    ),
    "all_valid_patients_have_oof_prediction": bool(
        oof_integrity["all_patients_covered"]
    ),
})
atomic_json_dump(
    final_summary,
    os.path.join(
        EXP_ROOT,
        "FINAL_nested_oof_summary.json",
    ),
)

print("\n" + "=" * 120)
print("FINAL NESTED OOF SUMMARY")
print("=" * 120)
print(json.dumps(final_summary, indent=2))
print("\nSaved outputs to:", EXP_ROOT)
print(
    "Main patient file:",
    os.path.join(
        EXP_ROOT,
        "FINAL_nested_oof_patient_predictions.csv",
    ),
)
print(
    "Main window file:",
    os.path.join(
        EXP_ROOT,
        "FINAL_nested_oof_window_predictions.csv",
    ),
)


# ============================================================
# 11B. Paired full-feature vs common-feature CNN comparison
# ============================================================
def bootstrap_common_minus_full_mae(paired_df, repeats=5000, seed=42):
    rng = np.random.default_rng(seed)
    delta = (
        paired_df["abs_error_common"].to_numpy(dtype=float)
        - paired_df["abs_error_full"].to_numpy(dtype=float)
    )
    n = len(delta)
    if n == 0:
        return np.nan, np.nan, np.nan
    estimates = np.empty(repeats, dtype=float)
    for i in range(repeats):
        idx = rng.integers(0, n, size=n)
        estimates[i] = float(np.mean(delta[idx]))
    return (
        float(np.mean(delta)),
        float(np.quantile(estimates, 0.025)),
        float(np.quantile(estimates, 0.975)),
    )


if os.path.isfile(MAIN_FULL_CNN_PATIENT_OOF_CSV):
    full_oof = pd.read_csv(MAIN_FULL_CNN_PATIENT_OOF_CSV, dtype={"pid": str})
    common_oof = final_patient_df.copy()
    full_oof["pid_key"] = full_oof["pid"].map(normalize_pid_text)
    common_oof["pid_key"] = common_oof["pid"].map(normalize_pid_text)
    full_pred_col = (
        "pred_patient_mean"
        if "pred_patient_mean" in full_oof.columns
        else "pred_patient"
    )
    common_pred_col = (
        "pred_patient_mean"
        if "pred_patient_mean" in common_oof.columns
        else "pred_patient"
    )
    paired_full_common = (
        full_oof[["pid_key", "true_patient", full_pred_col]]
        .rename(columns={
            "true_patient": "true_patient_full",
            full_pred_col: "pred_full",
        })
        .merge(
            common_oof[["pid_key", "true_patient", common_pred_col]],
            on="pid_key",
            how="inner",
            validate="one_to_one",
        )
        .rename(columns={
            "true_patient": "true_patient_common",
            common_pred_col: "pred_common",
        })
    )
    paired_full_common["true_difference_common_minus_full"] = (
        paired_full_common["true_patient_common"]
        - paired_full_common["true_patient_full"]
    )
    paired_full_common["abs_error_full"] = np.abs(
        paired_full_common["pred_full"]
        - paired_full_common["true_patient_full"]
    )
    paired_full_common["abs_error_common"] = np.abs(
        paired_full_common["pred_common"]
        - paired_full_common["true_patient_full"]
    )
    paired_full_common["abs_error_difference_common_minus_full"] = (
        paired_full_common["abs_error_common"]
        - paired_full_common["abs_error_full"]
    )
    delta, ci_low, ci_high = bootstrap_common_minus_full_mae(paired_full_common)
    paired_full_common.to_csv(
        os.path.join(EXP_ROOT, "FINAL_paired_full_vs_USA_common_CNN_patient_predictions.csv"),
        index=False,
    )
    comparison_summary = {
        "n_common_patients": int(len(paired_full_common)),
        "full_feature_patient_mae": float(paired_full_common["abs_error_full"].mean()),
        "usa_common_feature_patient_mae_using_full_truth": float(
            paired_full_common["abs_error_common"].mean()
        ),
        "delta_mae_common_minus_full": delta,
        "delta_mae_ci_low": ci_low,
        "delta_mae_ci_high": ci_high,
        "interpretation": "positive delta means common-feature CNN is worse",
        "max_abs_true_definition_difference": float(
            np.abs(paired_full_common["true_difference_common_minus_full"]).max()
        ),
    }
    atomic_json_dump(
        comparison_summary,
        os.path.join(EXP_ROOT, "FINAL_paired_full_vs_USA_common_CNN_summary.json"),
    )
    pd.DataFrame([comparison_summary]).to_csv(
        os.path.join(EXP_ROOT, "FINAL_paired_full_vs_USA_common_CNN_summary.csv"),
        index=False,
    )
else:
    warnings.warn(
        "Full-feature CNN OOF patient CSV was not found; paired full-vs-common "
        "comparison was skipped."
    )


# ============================================================
# 12. Compatibility aliases

# ============================================================
# 12. Compatibility aliases + post-analysis / plotting sections
# ============================================================
# These sections are adapted from your previous GA-bin, outlier trajectory,
# and MP shifted-tail analysis scripts, but use the new nested-CV outputs.

# Save old-style compatible filenames so older plotting snippets can also read them.
final_patient_alias = final_patient_df.copy()
if "abs_error_patient" not in final_patient_alias.columns:
    if "abs_error" in final_patient_alias.columns:
        final_patient_alias["abs_error_patient"] = final_patient_alias["abs_error"]
    else:
        final_patient_alias["abs_error_patient"] = np.abs(
            final_patient_alias["pred_patient_mean"] - final_patient_alias["true_patient"]
        )
final_patient_alias.to_csv(os.path.join(EXP_ROOT, "final_oof_predictions.csv"), index=False)
final_window_df.to_csv(os.path.join(EXP_ROOT, "final_window_oof_predictions.csv"), index=False)


def normalize_pid_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s


def possible_pid_keys(pid):
    keys = [pid, str(pid), normalize_pid_str(pid)]
    try:
        if not pd.isna(pid):
            keys.append(int(float(pid)))
            keys.append(str(int(float(pid))))
    except Exception:
        pass
    seen, out = set(), []
    for k in keys:
        marker = (type(k), str(k))
        if marker not in seen:
            out.append(k)
            seen.add(marker)
    return out


def pid_match_mask(df, col, pid):
    possible = possible_pid_keys(pid)
    possible_str = [normalize_pid_str(x) for x in possible]
    mask = df[col].isin(possible)
    mask = mask | df[col].astype(str).isin([str(x) for x in possible])
    mask = mask | df[col].apply(normalize_pid_str).isin(possible_str)
    return mask


def safe_filename_from_pid(pid):
    s = str(pid).replace("/", "_").replace("\\", "_").replace(" ", "_")
    s = re.sub(r"[^A-Za-z0-9_.-]", "_", s)
    return s


def find_feature_col(feature_cols, exact_candidates, contains_candidates=None):
    contains_candidates = contains_candidates or []
    for c in exact_candidates:
        if c in feature_cols:
            return c
    lower_map = {str(c).lower(): c for c in feature_cols}
    for c in exact_candidates:
        if str(c).lower() in lower_map:
            return lower_map[str(c).lower()]
    for key in contains_candidates:
        key = str(key).lower()
        matches = [c for c in feature_cols if key in str(c).lower()]
        if matches:
            return matches[0]
    return None


def extract_feature_from_full_raw_pkl(pid, feature_col):
    """Extract a patient-level static value such as GA or birthweight from the original full-feature X2."""
    if feature_col is None or feature_col not in feature_cols:
        return np.nan
    feature_idx = feature_cols.index(feature_col)
    for key in possible_pid_keys(pid):
        if key in X2_by_patient_raw:
            arr = X2_by_patient_raw.get(key, None)
            if arr is None:
                continue
            a = np.asarray(arr)
            if a.size > 0 and a.ndim == 4 and a.shape[-1] > feature_idx:
                return float(a[0, 0, 0, feature_idx])
    return np.nan


def assign_ga_group(ga):
    if pd.isna(ga):
        return np.nan
    if ga < 26:
        return "EP < 26 wk"
    elif 26 <= ga < 29:
        return "VP [26,29) wk"
    elif 29 <= ga < 32:
        return "MP [29,32) wk"
    elif 32 <= ga < 36:
        return "LP [32,36) wk"
    else:
        return np.nan


def safe_q25(x):
    x = np.asarray(x)
    return np.nan if len(x) == 0 else np.quantile(x, 0.25)


def safe_q75(x):
    x = np.asarray(x)
    return np.nan if len(x) == 0 else np.quantile(x, 0.75)


def add_ga_bw_columns(patient_df_in, window_df_in):
    ga_feature_col = find_feature_col(
        feature_cols,
        exact_candidates=["feats__ga_w", "ga", "ga_w", "gestational_age", "gestational_age_w"],
        contains_candidates=["ga"],
    )
    bw_feature_col = find_feature_col(
        feature_cols,
        exact_candidates=["feats__bw", "bw", "birth_weight", "birthweight", "birth_weight_g"],
        contains_candidates=["birth", "bw"],
    )
    print("\nPost-analysis feature columns:")
    print("GA feature:", ga_feature_col)
    print("BW feature:", bw_feature_col)

    patient_df_out = patient_df_in.copy()
    window_df_out = window_df_in.copy()

    pid_to_ga = {pid: extract_feature_from_full_raw_pkl(pid, ga_feature_col) for pid in patient_df_out["pid"].unique()}
    pid_to_bw = {pid: extract_feature_from_full_raw_pkl(pid, bw_feature_col) for pid in patient_df_out["pid"].unique()}

    for df in [patient_df_out, window_df_out]:
        df["ga_birth"] = df["pid"].map(pid_to_ga).round(3)
        df["birth_weight"] = df["pid"].map(pid_to_bw).round(0)
        df["ga_group"] = df["ga_birth"].apply(assign_ga_group)

    if "abs_error_patient" not in patient_df_out.columns:
        if "abs_error" in patient_df_out.columns:
            patient_df_out["abs_error_patient"] = patient_df_out["abs_error"]
        else:
            patient_df_out["abs_error_patient"] = np.abs(patient_df_out["pred_patient_mean"] - patient_df_out["true_patient"])

    return patient_df_out, window_df_out


final_patient_annotated, final_window_annotated = add_ga_bw_columns(final_patient_df, final_window_df)
final_patient_annotated.to_csv(os.path.join(EXP_ROOT, "FINAL_nested_oof_patient_predictions_with_ga_bw.csv"), index=False)
final_window_annotated.to_csv(os.path.join(EXP_ROOT, "FINAL_nested_oof_window_predictions_with_ga_bw.csv"), index=False)
# Also overwrite aliases with GA/BW-added versions for convenience.
final_patient_annotated.to_csv(os.path.join(EXP_ROOT, "final_oof_predictions.csv"), index=False)
final_window_annotated.to_csv(os.path.join(EXP_ROOT, "final_window_oof_predictions.csv"), index=False)


# ------------------------------------------------------------
# 12A. Highest-10%-error patient trajectory plots
# ------------------------------------------------------------
def run_percentile_outlier_analysis(
    patient_df_in,
    window_df_in,
    top_fraction=OUTLIER_TOP_FRACTION,
):
    if not 0.0 < top_fraction < 1.0:
        raise ValueError("top_fraction must be between 0 and 1")

    out_dir = os.path.join(EXP_ROOT, "outlier_patient_figures")
    individual_dir = os.path.join(out_dir, "individual_figures")
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(individual_dir, exist_ok=True)

    patient_df2 = patient_df_in.copy()
    window_df2 = window_df_in.copy()

    patient_df2["error_patient"] = patient_df2["pred_patient_mean"] - patient_df2["true_patient"]
    patient_df2["abs_error_patient"] = np.abs(patient_df2["error_patient"])

    n_outliers = max(
        1,
        int(math.ceil(len(patient_df2) * top_fraction)),
    )
    percentile_outliers = (
        patient_df2
        .nlargest(n_outliers, "abs_error_patient")
        .reset_index(drop=True)
    )
    threshold = float(
        percentile_outliers["abs_error_patient"].min()
    )
    percentile_outliers["outlier_rank"] = np.arange(
        1,
        len(percentile_outliers) + 1,
    )
    percentile_outliers["outlier_threshold_abs_error"] = threshold
    percentile_outliers["outlier_top_fraction"] = top_fraction

    percent_label = int(round(top_fraction * 100))
    outlier_csv = os.path.join(
        out_dir,
        f"top_{percent_label}percent_patient_outliers_with_ga_bw.csv",
    )
    percentile_outliers.to_csv(outlier_csv, index=False)

    print(
        f"\nPatient outliers = strict highest {top_fraction:.1%} "
        f"by absolute OOF error "
        f"(n={n_outliers})"
    )
    print(f"Threshold: abs_error_patient >= {threshold:.4f} weeks")
    print(
        f"Selected patients: {len(percentile_outliers)} / "
        f"{len(patient_df2)}"
    )
    print("Saved to:", outlier_csv)
    print(
        percentile_outliers[
            [
                "pid",
                "ga_birth",
                "birth_weight",
                "true_patient",
                "pred_patient_mean",
                "error_patient",
                "abs_error_patient",
                "n_windows",
            ]
        ].to_string(index=False)
    )

    plot_data = []
    for pid in percentile_outliers["pid"].tolist():
        patient_rows = patient_df2[pid_match_mask(patient_df2, "pid", pid)].copy()
        traj = window_df2[pid_match_mask(window_df2, "pid", pid)].copy()
        if len(patient_rows) == 0 or len(traj) == 0:
            continue
        row = patient_rows.iloc[0]
        traj = traj.sort_values("window_idx").reset_index(drop=True)
        plot_data.append({
            "pid": pid,
            "ga_birth": row.get("ga_birth", np.nan),
            "birth_weight": row.get("birth_weight", np.nan),
            "true_patient": row["true_patient"],
            "pred_patient": row["pred_patient_mean"],
            "error_patient": row["error_patient"],
            "abs_error_patient": row["abs_error_patient"],
            "traj": traj,
        })

    if not plot_data:
        print("No top-outlier trajectories found for plotting.")
        return

    n_patients = len(plot_data)
    n_cols = 3
    n_rows = int(np.ceil(n_patients / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)
    axes = axes.flatten()

    for ax, item in zip(axes, plot_data):
        traj = item["traj"]
        x = traj["window_idx"].values
        ax.plot(x, traj["true_window"].values, marker="o", linewidth=1.8, markersize=3, label="True PMA")
        ax.plot(x, traj["pred_window_mean"].values, marker="o", linewidth=1.8, markersize=3, label="Pred PMA")
        ax.set_title(
            f"PID: {item['pid']}\nGA={item['ga_birth']}, BW={item['birth_weight']} g, AbsErr={item['abs_error_patient']:.2f}",
            fontsize=10,
        )
        ax.set_xlabel("Window index")
        ax.set_ylabel("PMA")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(plot_data), len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Highest {top_fraction:.0%} Patient Errors: Window-level Trajectories", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    combined_fig_path = os.path.join(out_dir, f"top_{int(round(top_fraction * 100))}percent_outlier_trajectories_combined.png")
    fig.savefig(combined_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved combined outlier figure:", combined_fig_path)

    all_plot_rows = []
    for item in plot_data:
        traj = item["traj"].copy()
        x = traj["window_idx"].values
        fig = plt.figure(figsize=(8, 5))
        plt.plot(x, traj["true_window"].values, marker="o", linewidth=2, markersize=4, label="True PMA")
        plt.plot(x, traj["pred_window_mean"].values, marker="o", linewidth=2, markersize=4, label="Pred PMA")
        plt.title(
            f"PID: {item['pid']} | GA={item['ga_birth']} | BW={item['birth_weight']} g | "
            f"True={item['true_patient']:.2f} | Pred={item['pred_patient']:.2f} | AbsErr={item['abs_error_patient']:.2f}"
        )
        plt.xlabel("Window index")
        plt.ylabel("PMA")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        fig_path = os.path.join(individual_dir, f"trajectory_pid_{safe_filename_from_pid(item['pid'])}.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        traj["ga_birth"] = item["ga_birth"]
        traj["birth_weight"] = item["birth_weight"]
        traj["true_patient"] = item["true_patient"]
        traj["pred_patient"] = item["pred_patient"]
        traj["error_patient"] = item["error_patient"]
        traj["abs_error_patient"] = item["abs_error_patient"]
        all_plot_rows.append(traj)

    if all_plot_rows:
        combined_csv_path = os.path.join(out_dir, "all_plotted_outlier_trajectories.csv")
        pd.concat(all_plot_rows, axis=0, ignore_index=True).to_csv(combined_csv_path, index=False)
        print("Saved combined outlier trajectory CSV:", combined_csv_path)
    print("Saved individual outlier figures to:", individual_dir)


# ------------------------------------------------------------
# 12B. GA-bin analysis plots and tables
# ------------------------------------------------------------
def run_ga_bin_analysis(patient_df_in, window_df_in):
    ga_bin_dir = os.path.join(EXP_ROOT, "ga_bin_analysis")
    os.makedirs(ga_bin_dir, exist_ok=True)

    group_order = ["EP < 26 wk", "VP [26,29) wk", "MP [29,32) wk", "LP [32,36) wk"]
    group_colors = {
        "EP < 26 wk": "#b24726",
        "VP [26,29) wk": "#d49a27",
        "MP [29,32) wk": "#4c9a8a",
        "LP [32,36) wk": "#4a78c2",
    }
    group_labels_long = {
        "EP < 26 wk": "Extreme preterm (< 26 wk)",
        "VP [26,29) wk": "Very preterm ([26,29) wk)",
        "MP [29,32) wk": "Moderate preterm ([29,32) wk)",
        "LP [32,36) wk": "Late preterm ([32,36) wk)",
    }

    traj_df = window_df_in.dropna(subset=["ga_group"]).copy()
    patient_level_df = patient_df_in.dropna(subset=["ga_group"]).copy()

    if len(traj_df) == 0 or len(patient_level_df) == 0:
        print("GA-bin analysis skipped: no valid GA groups.")
        return

    traj_df.to_csv(os.path.join(ga_bin_dir, "ga_bin_trajectory.csv"), index=False)
    patient_level_df.to_csv(os.path.join(ga_bin_dir, "ga_bin_patient_level_summary.csv"), index=False)

    obs_mae = float(np.mean(np.abs(traj_df["pred_window_mean"] - traj_df["true_window"])))
    obs_bias = float(np.mean(traj_df["pred_window_mean"] - traj_df["true_window"]))
    obs_r2 = float(r2_score(traj_df["true_window"], traj_df["pred_window_mean"])) if traj_df["true_window"].nunique() > 1 else np.nan
    pat_mae = float(np.mean(np.abs(patient_level_df["pred_patient_mean"] - patient_level_df["true_patient"])))
    pat_bias = float(np.mean(patient_level_df["pred_patient_mean"] - patient_level_df["true_patient"]))
    pat_r2 = float(r2_score(patient_level_df["true_patient"], patient_level_df["pred_patient_mean"])) if patient_level_df["true_patient"].nunique() > 1 else np.nan

    # Combined scatter plot.
    fig = plt.figure(figsize=(8, 8))
    ax = plt.gca()
    for group in group_order:
        sub = traj_df[traj_df["ga_group"] == group]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["true_window"], sub["pred_window_mean"],
            s=18, alpha=0.20, color=group_colors[group], label=group,
        )
    mn = min(traj_df["true_window"].min(), traj_df["pred_window_mean"].min())
    mx = max(traj_df["true_window"].max(), traj_df["pred_window_mean"].max())
    xline = np.linspace(mn - 0.5, mx + 0.5, 300)
    ax.fill_between(xline, xline - 1, xline + 1, color="gray", alpha=0.12)
    ax.plot(xline, xline, color="gray", linewidth=1.8)
    if traj_df["true_window"].nunique() > 1:
        coef = np.polyfit(traj_df["true_window"], traj_df["pred_window_mean"], deg=1)
        ax.plot(xline, coef[0] * xline + coef[1], "k:", linewidth=1.5)
    txt = (
        f"Patient MAE = {pat_mae:.2f} wk\n"
        f"Window MAE = {obs_mae:.2f} wk\n"
        f"Patient $R^2$ = {pat_r2:.2f}\n"
        f"Window $R^2$ = {obs_r2:.2f}\n"
        f"Mean bias = {pat_bias:.2f} wk"
    )
    ax.text(0.97, 0.06, txt, transform=ax.transAxes, ha="right", va="bottom", fontsize=11,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
    ax.set_xlabel("Post-menstrual age (weeks)", fontsize=15)
    ax.set_ylabel("Estimated functional maturational age (weeks)", fontsize=15)
    ax.set_title("Nested CV model: PMA vs estimated FMA", fontsize=15)
    ax.legend(loc="upper left", framealpha=0.9, fontsize=11)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(mn - 0.3, mx + 0.3)
    ax.set_ylim(mn - 0.3, mx + 0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(ga_bin_dir, "01_ga_bin_combined_scatter.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Combined binned median trajectory + patient count.
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 9), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08})
    all_min = min(traj_df["true_window"].min(), traj_df["pred_window_mean"].min())
    all_max = max(traj_df["true_window"].max(), traj_df["pred_window_mean"].max())
    ref_x = np.linspace(all_min - 0.5, all_max + 0.5, 300)
    ax1.plot(ref_x, ref_x, linestyle="--", color="black", linewidth=1.5, label="Reference (FMA = PMA)")

    group_summary_rows = []
    count_summary_all = []

    for group in group_order:
        sub = traj_df[traj_df["ga_group"] == group].copy()
        if len(sub) == 0:
            continue
        pmin, pmax = sub["true_window"].min(), sub["true_window"].max()
        bins = np.arange(np.floor(pmin * 2) / 2, np.ceil(pmax * 2) / 2 + 0.51, 0.5)
        if len(bins) < 2:
            continue
        sub["pma_bin"] = pd.cut(sub["true_window"], bins=bins, include_lowest=True)
        grouped = (
            sub.groupby("pma_bin", observed=True)
            .agg(
                true_mid=("true_window", "median"),
                pred_med=("pred_window_mean", "median"),
                pred_q1=("pred_window_mean", safe_q25),
                pred_q3=("pred_window_mean", safe_q75),
                n_obs=("pred_window_mean", "size"),
                n_patients=("pid", "nunique"),
            )
            .reset_index(drop=True)
        ).dropna(subset=["true_mid", "pred_med", "pred_q1", "pred_q3"])
        if len(grouped) == 0:
            continue
        color = group_colors[group]
        ax1.plot(grouped["true_mid"], grouped["pred_med"], color=color, linewidth=2.5,
                 label=group_labels_long[group])
        ax1.fill_between(grouped["true_mid"], grouped["pred_q1"], grouped["pred_q3"], color=color, alpha=0.12)

        tmp_count = grouped[["true_mid", "n_patients", "n_obs"]].copy()
        tmp_count["ga_group"] = group
        tmp_count["label"] = group_labels_long[group]
        count_summary_all.append(tmp_count)

        pat_sub = patient_level_df[patient_level_df["ga_group"] == group]
        if len(pat_sub) >= 1:
            g_pat_mae = float(np.mean(np.abs(pat_sub["pred_patient_mean"] - pat_sub["true_patient"])))
            g_pat_bias = float(np.mean(pat_sub["pred_patient_mean"] - pat_sub["true_patient"]))
            g_pat_r2 = float(r2_score(pat_sub["true_patient"], pat_sub["pred_patient_mean"])) if pat_sub["true_patient"].nunique() > 1 else np.nan
        else:
            g_pat_mae, g_pat_bias, g_pat_r2 = np.nan, np.nan, np.nan
        group_summary_rows.append({
            "ga_group": group,
            "label": group_labels_long[group],
            "n_patients": int(pat_sub["pid"].nunique()),
            "n_windows": int(len(sub)),
            "patient_mae": g_pat_mae,
            "patient_r2": g_pat_r2,
            "patient_bias": g_pat_bias,
        })

    ax1.set_ylabel("Estimated functional maturational age (weeks)", fontsize=14)
    ax1.set_title("Binned median FMA trajectories by gestational age category", fontsize=15)
    ax1.legend(loc="upper left", framealpha=0.9, fontsize=10)
    ax1.grid(True, alpha=0.25)

    if count_summary_all:
        count_summary_df = pd.concat(count_summary_all, ignore_index=True)
        count_summary_df.to_csv(os.path.join(ga_bin_dir, "ga_bin_patient_count_over_pma.csv"), index=False)
        for group in group_order:
            sub_count = count_summary_df[count_summary_df["ga_group"] == group]
            if len(sub_count) == 0:
                continue
            color = group_colors[group]
            ax2.plot(sub_count["true_mid"], sub_count["n_patients"], color=color, linewidth=2.0, label=group)
            ax2.scatter(sub_count["true_mid"], sub_count["n_patients"], color=color, s=18, alpha=0.8)

    ax2.set_xlabel("Post-menstrual age (weeks)", fontsize=14)
    ax2.set_ylabel("Patient count", fontsize=13)
    ax2.grid(True, alpha=0.25)
    ax2.legend(loc="upper right", framealpha=0.9, fontsize=9)
    ax2.set_xlim(all_min - 0.3, all_max + 0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(ga_bin_dir, "02_ga_bin_binned_median_trajectory_with_count.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Four-panel trajectory version.
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    axes = axes.flatten()
    fig.suptitle("Individual and median FMA trajectories by gestational age category", fontsize=16)
    panel_summary_rows = []

    for ax, group in zip(axes, group_order):
        sub = traj_df[traj_df["ga_group"] == group].copy()
        color = group_colors[group]
        if len(sub) == 0:
            ax.set_title(group_labels_long[group], fontsize=13, fontweight="bold")
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.grid(True, alpha=0.25)
            continue
        for pid, d in sub.groupby("pid"):
            d = d.sort_values("true_window")
            ax.plot(d["true_window"].values, d["pred_window_mean"].values, color=color, alpha=0.05, linewidth=1)
        pmin, pmax = sub["true_window"].min(), sub["true_window"].max()
        bins = np.arange(np.floor(pmin * 2) / 2, np.ceil(pmax * 2) / 2 + 0.51, 0.5)
        if len(bins) >= 2:
            sub["pma_bin"] = pd.cut(sub["true_window"], bins=bins, include_lowest=True)
            grouped = (
                sub.groupby("pma_bin", observed=True)
                .agg(
                    true_mid=("true_window", "median"),
                    pred_med=("pred_window_mean", "median"),
                    pred_q1=("pred_window_mean", safe_q25),
                    pred_q3=("pred_window_mean", safe_q75),
                )
            ).dropna(subset=["true_mid", "pred_med", "pred_q1", "pred_q3"])
            if len(grouped) > 0:
                ax.fill_between(grouped["true_mid"], grouped["pred_q1"], grouped["pred_q3"], color=color, alpha=0.12)
                ax.plot(grouped["true_mid"], grouped["pred_med"], color=color, linewidth=2.5)
        mn_g = min(sub["true_window"].min(), sub["pred_window_mean"].min())
        mx_g = max(sub["true_window"].max(), sub["pred_window_mean"].max())
        ax.plot([mn_g, mx_g], [mn_g, mx_g], "--", color="black", linewidth=1)

        pat_sub = patient_level_df[patient_level_df["ga_group"] == group]
        if len(pat_sub) >= 1:
            g_pat_mae = float(np.mean(np.abs(pat_sub["pred_patient_mean"] - pat_sub["true_patient"])))
            g_pat_bias = float(np.mean(pat_sub["pred_patient_mean"] - pat_sub["true_patient"]))
            g_pat_r2 = float(r2_score(pat_sub["true_patient"], pat_sub["pred_patient_mean"])) if pat_sub["true_patient"].nunique() > 1 else np.nan
        else:
            g_pat_mae, g_pat_bias, g_pat_r2 = np.nan, np.nan, np.nan
        n_pat = int(pat_sub["pid"].nunique())
        ax.text(
            0.97, 0.05,
            f"Pat MAE = {g_pat_mae:.2f} wk\nPat $R^2$ = {g_pat_r2:.2f}\nBias = {g_pat_bias:.2f} wk\nn = {n_pat} patients",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75),
        )
        ax.set_title(group_labels_long[group], fontsize=13, fontweight="bold")
        ax.set_xlabel("PMA (weeks)", fontsize=12)
        ax.set_ylabel("FMA (weeks)", fontsize=12)
        ax.grid(True, alpha=0.25)
        panel_summary_rows.append({
            "ga_group": group,
            "label": group_labels_long[group],
            "n_patients": n_pat,
            "n_windows": int(len(sub)),
            "patient_mae": g_pat_mae,
            "patient_r2": g_pat_r2,
            "patient_bias": g_pat_bias,
        })

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(os.path.join(ga_bin_dir, "03_ga_bin_4panel_trajectory.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    pd.DataFrame(group_summary_rows).to_csv(os.path.join(ga_bin_dir, "ga_bin_group_summary.csv"), index=False)
    pd.DataFrame(panel_summary_rows).to_csv(os.path.join(ga_bin_dir, "ga_bin_panel_summary.csv"), index=False)
    pd.DataFrame([{
        "observation_mae": obs_mae,
        "observation_r2": obs_r2,
        "observation_bias": obs_bias,
        "patient_mae": pat_mae,
        "patient_r2": pat_r2,
        "patient_bias": pat_bias,
        "n_observations": len(traj_df),
        "n_patients": int(patient_level_df["pid"].nunique()),
    }]).to_csv(os.path.join(ga_bin_dir, "ga_bin_overall_metrics.csv"), index=False)
    print("GA-bin analysis saved to:", ga_bin_dir)


# ------------------------------------------------------------
# 12C. Identify MP shifted-tail patients
# ------------------------------------------------------------
def run_mp_shifted_tail_analysis(window_df_in, high_pma_threshold=36.0, top_label_n=10):
    out_dir = os.path.join(EXP_ROOT, "identify_shifted_mp_patient")
    os.makedirs(out_dir, exist_ok=True)
    individual_dir = os.path.join(out_dir, "individual_suspect_patients")
    os.makedirs(individual_dir, exist_ok=True)

    traj_df = window_df_in.copy()
    if "error_window" not in traj_df.columns:
        traj_df["error_window"] = traj_df["pred_window_mean"] - traj_df["true_window"]
    traj_df["abs_error_window"] = np.abs(traj_df["error_window"])

    mp_df = traj_df[traj_df["ga_group"] == "MP [29,32) wk"].copy()
    if len(mp_df) == 0:
        print("MP shifted-tail analysis skipped: no MP [29,32) wk windows found.")
        return

    mp_high = mp_df[mp_df["true_window"] >= high_pma_threshold].copy()
    if len(mp_high) == 0:
        print(f"MP shifted-tail analysis skipped: no MP windows with true PMA >= {high_pma_threshold}.")
        return

    suspect_summary = (
        mp_high.groupby("pid", as_index=False)
        .agg(
            ga_birth=("ga_birth", "first"),
            birth_weight=("birth_weight", "first"),
            n_tail_windows=("true_window", "size"),
            true_min_tail=("true_window", "min"),
            true_max_tail=("true_window", "max"),
            true_mean_tail=("true_window", "mean"),
            pred_min_tail=("pred_window_mean", "min"),
            pred_max_tail=("pred_window_mean", "max"),
            pred_mean_tail=("pred_window_mean", "mean"),
            mean_error_tail=("error_window", "mean"),
            mean_abs_error_tail=("abs_error_window", "mean"),
            max_abs_error_tail=("abs_error_window", "max"),
        )
        .reset_index(drop=True)
    )
    full_summary = (
        mp_df.groupby("pid", as_index=False)
        .agg(
            n_total_windows=("true_window", "size"),
            true_min_all=("true_window", "min"),
            true_max_all=("true_window", "max"),
            pred_min_all=("pred_window_mean", "min"),
            pred_max_all=("pred_window_mean", "max"),
            mean_error_all=("error_window", "mean"),
            mean_abs_error_all=("abs_error_window", "mean"),
        )
    )
    suspect_summary = suspect_summary.merge(full_summary, on="pid", how="left")
    suspect_summary = suspect_summary.sort_values(
        ["n_tail_windows", "mean_error_tail", "true_max_tail"], ascending=[False, True, False]
    ).reset_index(drop=True)
    suspect_path = os.path.join(out_dir, "suspect_mp_patients_high_pma_tail.csv")
    suspect_summary.to_csv(suspect_path, index=False)
    print("MP shifted-tail suspect table saved to:", suspect_path)

    range_summary = (
        mp_df.groupby("pid", as_index=False)
        .agg(
            ga_birth=("ga_birth", "first"),
            birth_weight=("birth_weight", "first"),
            n_windows=("true_window", "size"),
            true_min=("true_window", "min"),
            true_max=("true_window", "max"),
            pred_min=("pred_window_mean", "min"),
            pred_max=("pred_window_mean", "max"),
            mean_error=("error_window", "mean"),
            mean_abs_error=("abs_error_window", "mean"),
        )
    )
    range_summary["true_pma_range"] = range_summary["true_max"] - range_summary["true_min"]
    range_summary["pred_pma_range"] = range_summary["pred_max"] - range_summary["pred_min"]
    range_summary = range_summary.sort_values(["true_max", "true_pma_range", "mean_abs_error"], ascending=[False, False, False]).reset_index(drop=True)
    range_summary.to_csv(os.path.join(out_dir, "mp_patients_ranked_by_true_pma_range.csv"), index=False)

    label_pids = suspect_summary["pid"].head(top_label_n).tolist()

    fig = plt.figure(figsize=(9, 7))
    ax = plt.gca()
    for pid, sub in mp_df.groupby("pid"):
        sub = sub.sort_values("true_window")
        ax.plot(sub["true_window"], sub["pred_window_mean"], linewidth=0.8, alpha=0.15)
    for pid in label_pids:
        sub = mp_df[pid_match_mask(mp_df, "pid", pid)].copy()
        if len(sub) == 0:
            continue
        sub = sub.sort_values("true_window")
        ax.plot(sub["true_window"], sub["pred_window_mean"], marker="o", linewidth=2.2, markersize=3, label=f"pid={pid}")
        last = sub.iloc[-1]
        ax.text(last["true_window"], last["pred_window_mean"], f" {pid}", fontsize=8, va="center")
    mn = min(mp_df["true_window"].min(), mp_df["pred_window_mean"].min())
    mx = max(mp_df["true_window"].max(), mp_df["pred_window_mean"].max())
    ax.plot([mn, mx], [mn, mx], "--", linewidth=1.2)
    ax.axvline(high_pma_threshold, linestyle=":", linewidth=1.2)
    ax.set_xlabel("True PMA / window-level PMA")
    ax.set_ylabel("Predicted FMA / predicted PMA")
    ax.set_title(
        f"MP [29,32) wk: likely patients causing high-PMA shifted tail\n"
        f"Labelled: top {top_label_n} patients with true PMA >= {high_pma_threshold}"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="best")
    fig.tight_layout()
    fig_path = os.path.join(out_dir, "mp_group_suspect_patient_trajectories_labeled.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved labelled MP shifted-tail figure:", fig_path)

    for pid in label_pids:
        sub = mp_df[pid_match_mask(mp_df, "pid", pid)].copy()
        if len(sub) == 0:
            continue
        sub = sub.sort_values("window_idx").reset_index(drop=True)
        x = sub["window_idx"].values
        ga = sub["ga_birth"].iloc[0]
        bw = sub["birth_weight"].iloc[0]
        fig = plt.figure(figsize=(8, 5))
        ax = plt.gca()
        ax.plot(x, sub["true_window"], marker="o", linewidth=2, markersize=4, label="True PMA")
        ax.plot(x, sub["pred_window_mean"], marker="o", linewidth=2, markersize=4, label="Pred FMA")
        ax.set_title(
            f"PID={pid} | GA={ga} | BW={bw} g\n"
            f"True PMA {sub['true_window'].min():.2f}-{sub['true_window'].max():.2f}, "
            f"Mean error={sub['error_window'].mean():.2f} wk"
        )
        ax.set_xlabel("window_idx")
        ax.set_ylabel("Weeks")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(individual_dir, f"suspect_pid_{safe_filename_from_pid(pid)}_trajectory.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)
    print("Saved individual MP suspect patient figures to:", individual_dir)


if RUN_POST_ANALYSIS:
    print("\n" + "=" * 120)
    print("RUNNING POST-ANALYSIS SECTIONS")
    print("=" * 120)
    run_percentile_outlier_analysis(
        final_patient_annotated,
        final_window_annotated,
        top_fraction=OUTLIER_TOP_FRACTION,
    )
    run_ga_bin_analysis(final_patient_annotated, final_window_annotated)
    run_mp_shifted_tail_analysis(
        final_window_annotated,
        high_pma_threshold=36.0,
        top_label_n=10,
    )

print("\nAll training and post-analysis steps finished.")
print("EXP_ROOT:", EXP_ROOT)

Device: cuda
Normalization mode: patient_balanced_per_dynamic_feature_zscore_training_fold_only
SOURCE_DATA_PKL: /mnt/home/qinqiu/thesis/data/superwindow_stride_comparison_90_120_3h_no_repeated_weight_no_los_nec_ga_le_35_no_intubation_imputed_2ch_CTHW/data_patients_superwindow_120min_stride40min_overlap67_timebased_featsQC_loose80_imputed_2ch_CTHW.pkl
COMMON_DATA_PKL: /mnt/home/qinqiu/thesis/data/sweden_usa_common_feature_120min_stride40min/sweden_120min_stride40min_usa_common17_imputed_2ch_CTHW.pkl
EXP_ROOT: /mnt/home/qinqiu/thesis/result_nested/nested_cv_120min_stride40min_usa_common17_v1_patientbalanced_resume100
Common-feature PKL created: /mnt/home/qinqiu/thesis/data/sweden_usa_common_feature_120min_stride40min/sweden_120min_stride40min_usa_common17_imputed_2ch_CTHW.pkl

Common feature structure:
Example pid: 021de38ed3468b4c98d76a19c3e4cd1f9b537e3aa6923efe7ee93db6991bdd89
Example X shape before static separation: (7, 2, 23, 17)
Dynamic features: ['feats__spo2_mean', 'feats__spo2_

[I 2026-07-27 19:11:00,047] A new study created in RDB with name: nested_arch_optuna_outer_01


Optuna outer 1 resume status | COMPLETE=0, PRUNED=0, FAIL=0, RUNNING=0, WAITING=0, COUNTED=0/100, REMAINING=100


[I 2026-07-27 19:18:27,952] Trial 0 finished with value: 0.8394079918852704 and parameters: {'architecture': 'A2_16_32_pool_simple_head', 'static_dim': 8, 'dropout': 0.36687414814014147, 'batch_size': 64, 'optimizer': 'AdamW', 'lr': 0.00023742236901921776, 'weight_decay': 1.65263259389039e-05, 'smooth_l1_beta': 2.0, 'grad_clip_norm': 1.0, 'patience': 15, 'min_delta': 0.0001, 'scheduler_type': 'cosine', 'cosine_eta_min_ratio': 0.05}. Best is trial 0 with value: 0.8394079918852704.
[I 2026-07-27 19:24:25,215] Trial 1 finished with value: 0.7958345736310782 and parameters: {'architecture': 'A2_16_32_pool_simple_head', 'static_dim': 8, 'dropout': 0.3143928979410806, 'batch_size': 64, 'optimizer': 'AdamW', 'lr': 0.0017781903246150339, 'weight_decay': 2.739399183644026e-05, 'smooth_l1_beta': 2.0, 'grad_clip_norm': 5.0, 'patience': 15, 'min_delta': 0.0001, 'scheduler_type': 'none'}. Best is trial 1 with value: 0.7958345736310782.
[I 2026-07-27 19:28:53,904] Trial 2 finished with value: 0.8974

Optuna outer 1 resume status | COMPLETE=99, PRUNED=1, FAIL=0, RUNNING=0, WAITING=0, COUNTED=100/100, REMAINING=0
Best outer 1: trial=61, inner MAE=0.7173
Best params: {'architecture': 'A6_16_32_64_pool_simple_head', 'static_dim': 8, 'dropout': 0.008248772047889671, 'batch_size': 64, 'optimizer': 'Adam', 'lr': 0.0008060671577717737, 'weight_decay': 0.00024356445612559163, 'smooth_l1_beta': 0.5, 'grad_clip_norm': 0.0, 'patience': 30, 'min_delta': 0.001, 'scheduler_type': 'none', 'head_dim': 0, 'cosine_eta_min_ratio': None}
Final refit epochs: 36
[outer1_FULL_seed42] epoch=001/036 train_loss=8.2319 lr=8.061e-04
[outer1_FULL_seed42] epoch=010/036 train_loss=0.7354 lr=8.061e-04
[outer1_FULL_seed42] epoch=020/036 train_loss=0.6999 lr=8.061e-04
[outer1_FULL_seed42] epoch=030/036 train_loss=0.6613 lr=8.061e-04
[outer1_FULL_seed42] epoch=036/036 train_loss=0.6536 lr=8.061e-04
[outer1_FULL_seed123] epoch=001/036 train_loss=9.1951 lr=8.061e-04
[outer1_FULL_seed123] epoch=010/036 train_loss=0.7415

...

[I 2026-07-28 22:35:00,594] Trial 0 finished with value: 0.8867364527314955 and parameters: {'architecture': 'A1_8_16_pool_simple_head', 'static_dim': 8, 'dropout': 0.313695841352213, 'batch_size': 128, 'optimizer': 'AdamW', 'lr': 0.0003021642672587241, 'weight_decay': 0.02216606006505342, 'smooth_l1_beta': 0.5, 'grad_clip_norm': 0.0, 'patience': 30, 'min_delta': 0.0005, 'scheduler_type': 'none'}. Best is trial 0 with value: 0.8867364527314955.
[I 2026-07-28 22:45:39,950] Trial 1 finished with value: 0.8159698679147521 and parameters: {'architecture': 'A4_32_64_pool_simple_head', 'static_dim': 64, 'dropout': 0.15503696875811118, 'batch_size': 32, 'optimizer': 'Adam', 'lr': 0.00016112364515936706, 'weight_decay': 0.004871903571045122, 'smooth_l1_beta': 1.0, 'grad_clip_norm': 0.0, 'patience': 20, 'min_delta': 0.0005, 'scheduler_type': 'cosine', 'cosine_eta_min_ratio': 0.001}. Best is trial 1 with value: 0.8159698679147521.
[I 2026-07-28 22:59:35,270] Trial 2 finished with value: 0.856303

Optuna outer 3 resume status | COMPLETE=48, PRUNED=52, FAIL=0, RUNNING=0, WAITING=0, COUNTED=100/100, REMAINING=0
Best outer 3: trial=20, inner MAE=0.7462
Best params: {'architecture': 'A6_16_32_64_pool_simple_head', 'static_dim': 64, 'dropout': 0.2506148055041648, 'batch_size': 64, 'optimizer': 'AdamW', 'lr': 0.004863119794887775, 'weight_decay': 0.0003667729828197201, 'smooth_l1_beta': 2.0, 'grad_clip_norm': 5.0, 'patience': 20, 'min_delta': 0.0001, 'scheduler_type': 'none', 'head_dim': 0, 'cosine_eta_min_ratio': None}
Final refit epochs: 33
[outer3_FULL_seed42] epoch=001/033 train_loss=2.4571 lr=4.863e-03
[outer3_FULL_seed42] epoch=010/033 train_loss=0.8216 lr=4.863e-03
[outer3_FULL_seed42] epoch=020/033 train_loss=0.6030 lr=4.863e-03
[outer3_FULL_seed42] epoch=030/033 train_loss=0.4550 lr=4.863e-03
[outer3_FULL_seed42] epoch=033/033 train_loss=0.4151 lr=4.863e-03
[outer3_FULL_seed123] epoch=001/033 train_loss=2.4321 lr=4.863e-03
[outer3_FULL_seed123] epoch=010/033 train_loss=0.8321

[I 2026-07-29 07:21:02,404] A new study created in RDB with name: nested_arch_optuna_outer_04


Optuna outer 4 resume status | COMPLETE=0, PRUNED=0, FAIL=0, RUNNING=0, WAITING=0, COUNTED=0/100, REMAINING=100


[I 2026-07-29 07:31:18,243] Trial 0 finished with value: 0.9260536563852445 and parameters: {'architecture': 'A1_8_16_pool_simple_head', 'static_dim': 8, 'dropout': 0.2275019594633894, 'batch_size': 32, 'optimizer': 'AdamW', 'lr': 0.00035473050023150627, 'weight_decay': 3.715592637072014e-07, 'smooth_l1_beta': 0.5, 'grad_clip_norm': 1.0, 'patience': 25, 'min_delta': 0.001, 'scheduler_type': 'cosine', 'cosine_eta_min_ratio': 0.01}. Best is trial 0 with value: 0.9260536563852445.
